# 05 HRV Domain And Robustness

Reviewer-facing HRV-domain and robustness readiness/status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

# Keep all git operations out of Drive-backed cwd. This avoids Colab
# "Transport endpoint is not connected" failures after Drive hiccups.
os.chdir('/content')
print('cwd reset:', Path.cwd())

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or was stale: {exc}')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.')

REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
CANONICAL_CHECKPOINT_ROOT = MIRROR_REVISION_ROOT / 'experimental'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')
REPO_DIR = Path('/content/ECG-RAMBA')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd='/content', check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _is_repo(path):
    path = Path(path)
    return (path / '.git').exists() and (path / 'configs' / 'config.py').exists()

def _remove_local_repo(reason):
    if REPO_DIR.exists():
        print(f'Removing local repo at {REPO_DIR}: {reason}')
        _run_setup(f'rm -rf "{REPO_DIR}"', cwd='/content')

if REPO_DIR.exists() and not _is_repo(REPO_DIR):
    _remove_local_repo('path exists but is not a valid repo')

if not REPO_DIR.exists():
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')
else:
    print('Using existing local repo:', REPO_DIR)

lock_path = REPO_DIR / '.git' / 'index.lock'
if lock_path.exists():
    print('Removing stale git index lock:', lock_path)
    lock_path.unlink()

fetch_result = _run_setup('git fetch origin', cwd=REPO_DIR, check=False)
checkout_result = _run_setup(f'git checkout {BRANCH}', cwd=REPO_DIR, check=False)
if fetch_result.returncode == 0 and checkout_result.returncode == 0:
    pull_result = _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR, check=False)
    if pull_result.returncode:
        print('git pull failed; replacing /content clone with a fresh checkout.')
        _remove_local_repo('pull failed')
        _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')
else:
    print('git fetch/checkout failed; replacing /content clone with a fresh checkout.')
    _remove_local_repo('fetch or checkout failed')
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')

os.chdir(REPO_DIR)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_versioned_git_release_v2'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 2
_AUTHORITY_BOOTSTRAP_ALLOWED = False
_AUTHORITY_DEFAULT_RELEASE_REF = 'refs/tags/ecg-ramba-revision-20260722-v6'

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath
    from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    requested_release_ref = _authority_os.environ.get(
        'ECG_RAMBA_AUTHORITY_REF', _AUTHORITY_DEFAULT_RELEASE_REF
    ).strip()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    legacy_rotate_requested = (
        _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
    )
    if legacy_rotate_requested:
        raise RuntimeError(
            'Implicit authority rotation to a moving branch head is disabled. '
            'Notebook 00 follows only a reviewed versioned release tag. For an emergency explicit '
            'pin, set ECG_RAMBA_RESET_CODE_AUTHORITY=1 with a full ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    rotate_to_branch_head = False
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')
    release_ref_pattern = _authority_re.compile(r'refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    def resolve_annotated_release_ref(release_ref):
        if not release_ref_pattern.fullmatch(release_ref):
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_REF must be a full refs/tags/... name. '
                'Moving branches are not accepted as code authority.'
            )
        object_type = git('cat-file', '-t', release_ref, check=False)
        if object_type.returncode or object_type.stdout.strip() != 'tag':
            raise RuntimeError(
                f'Code-authority release ref is missing or is not an annotated Git tag: {release_ref}. '
                'Run the current Notebook 00 after the reviewed release tag has been pushed.'
            )
        object_id = git('rev-parse', '--verify', release_ref).stdout.strip().lower()
        commit = git('rev-parse', '--verify', release_ref + '^{commit}').stdout.strip().lower()
        if not commit_pattern.fullmatch(object_id) or not commit_pattern.fullmatch(commit):
            raise RuntimeError(f'Could not resolve an immutable object and commit for {release_ref}.')
        return commit, object_id

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', '--tags', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit/release tag.')
        print((fetch.stdout or '')[-2000:])

    manifest = None
    manifest_raw = None
    with _AuthorityPublishLock(
        _AuthorityPath(MIRROR_REVISION_ROOT),
        run_id='authority-read-' + _authority_uuid.uuid4().hex,
    ):
        if manifest_path.is_file():
            manifest_raw = manifest_path.read_text(encoding='utf-8')
            manifest = _authority_json.loads(manifest_raw)

    authority_update_needed = False
    update_reason = None
    release_ref = None
    release_ref_object_id = None

    if reset_requested:
        expected_commit = requested_commit
        authority_update_needed = True
        update_reason = 'explicit_environment_sha'
    elif manifest is None:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        release_ref = requested_release_ref
        expected_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
        authority_update_needed = True
        update_reason = 'initial_versioned_release_bootstrap'
    else:
        manifest_capability = manifest.get('capability')
        manifest_schema = int(manifest.get('schema_version', 0))
        legacy_manifest = (
            manifest_capability == 'canonical_git_commit_pin_v1' and manifest_schema == 1
        )
        current_manifest = (
            manifest_capability == 'canonical_versioned_git_release_v2' and manifest_schema == 2
        )
        if not legacy_manifest and not current_manifest:
            raise RuntimeError('Canonical code-authority manifest capability/schema is invalid.')
        manifest_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(manifest_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')

        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            if legacy_manifest:
                raise RuntimeError(
                    'Canonical code authority uses the legacy schema. Run the current Notebook 00 once '
                    'to migrate it to the reviewed versioned release before running downstream notebooks.'
                )
            expected_commit = manifest_commit
            if requested_commit and requested_commit != expected_commit:
                raise RuntimeError(
                    'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                    'Rotate authority explicitly in Notebook 00; do not override it downstream.'
                )
        else:
            release_ref = requested_release_ref
            release_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
            manifest_ref = str(manifest.get('authority_ref', '')).strip()
            manifest_ref_object = str(manifest.get('authority_ref_object_id', '')).strip().lower()
            if current_manifest and manifest_ref == release_ref:
                if manifest_commit != release_commit or manifest_ref_object != release_ref_object_id:
                    raise RuntimeError(
                        f'Code-authority release tag moved or changed after it was recorded: {release_ref}. '
                        'Publish a new versioned tag instead of retagging an existing release.'
                    )
                expected_commit = manifest_commit
            else:
                expected_commit = release_commit
                authority_update_needed = True
                update_reason = (
                    'legacy_manifest_migration'
                    if legacy_manifest
                    else 'versioned_release_upgrade'
                )

    if not commit_pattern.fullmatch(expected_commit):
        raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if authority_update_needed:
        previous_commit = None if manifest is None else str(manifest.get('git_commit', '')).strip().lower()
        previous_ref = None if manifest is None else manifest.get('authority_ref')
        manifest = {
            'capability': 'canonical_versioned_git_release_v2',
            'schema_version': 2,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'explicit_environment_sha'
                if release_ref is None
                else 'verified_annotated_versioned_release_tag'
            ),
            'authority_ref': release_ref,
            'authority_ref_kind': None if release_ref is None else 'annotated_git_tag',
            'authority_ref_object_id': release_ref_object_id,
            'update_reason': update_reason,
            'previous_git_commit': previous_commit,
            'previous_authority_ref': previous_ref,
        }
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-write-' + _authority_uuid.uuid4().hex,
        ):
            concurrent_raw = manifest_path.read_text(encoding='utf-8') if manifest_path.is_file() else None
            if concurrent_raw != manifest_raw and not reset_requested:
                concurrent = _authority_json.loads(concurrent_raw) if concurrent_raw else None
                if not (
                    concurrent
                    and concurrent.get('capability') == 'canonical_versioned_git_release_v2'
                    and int(concurrent.get('schema_version', 0)) == 2
                    and str(concurrent.get('git_commit', '')).lower() == expected_commit
                    and concurrent.get('authority_ref') == release_ref
                    and str(concurrent.get('authority_ref_object_id', '')).lower()
                    == str(release_ref_object_id or '').lower()
                ):
                    raise RuntimeError('A concurrent runtime established a different code authority.')
                manifest = concurrent
            else:
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = manifest_path.with_name(
                    manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                )
                with temporary.open('w', encoding='utf-8') as handle:
                    handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                    handle.flush()
                    _authority_os.fsync(handle.fileno())
                _authority_os.replace(temporary, manifest_path)
        print('Established/updated canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority release  :', manifest.get('authority_ref'))
    print('Authority manifest :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
_run_setup('git rev-parse --short HEAD', cwd=REPO_DIR, check=False)
_run_setup('git status --short --branch', cwd=REPO_DIR, check=False)

# Direct-run bootstrap: Notebook 05 no longer depends on Notebook 00 for base Python packages.
# Set ECG_RAMBA_INSTALL_NOTEBOOK05_BASE_DEPS=0 only when the runtime has already been prepared.
INSTALL_NOTEBOOK05_BASE_DEPS = os.environ.get('ECG_RAMBA_INSTALL_NOTEBOOK05_BASE_DEPS', '1') == '1'
if INSTALL_NOTEBOOK05_BASE_DEPS:
    base_dep_command = (
        'python -m pip install -q '
        '"numpy>=2.0,<2.1" "scipy>=1.14.1,<2.0" "pandas==2.2.2" '
        '"scikit-learn>=1.2,<1.9" "requests==2.32.4" "pillow>=8,<12" '
        'tqdm "wfdb==4.1.2" joblib matplotlib seaborn packaging neurokit2 '
        'iterative-stratification thop'
    )
    _run_setup(base_dep_command, cwd=REPO_DIR, check=True)
else:
    print('Skipping Notebook 05 base dependency install because ECG_RAMBA_INSTALL_NOTEBOOK05_BASE_DEPS=0')


REQUIRED_REVISION_ARTIFACTS_FOR_05 = [
    'manifests/oof_final_ema_freeze_manifest.json',
    'manifests/oof_final_ema_prediction_run_manifest.json',
    'predictions/oof_final_ema_predictions.npz',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'metrics/baseline_summary.csv',
    'metrics/component_check_summary.json',
    'predictions/resnet1d_cnn_oof_predictions.npz',
    'predictions/raw_mamba_oof_predictions.npz',
    'predictions/transformer_ecg_oof_predictions.npz',
    'predictions/robustness_minirocket_clean_ref_predictions.npz',
    'manifests/resnet1d_cnn_baseline_manifest.json',
    'manifests/raw_mamba_baseline_manifest.json',
    'manifests/transformer_ecg_baseline_manifest.json',
    'manifests/robustness_minirocket_heads_manifest.json',
    'audit_protocol.json',
    'a0_resolution_status.json',
    'hrv36_schema.csv',
]


def _sha256_local(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _mirror_manifest_rows(mirror_root):
    manifest_path = Path(mirror_root) / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.exists():
        return manifest_path, {}
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    declared_root = Path(payload.get('mirror_root', mirror_root))
    rows = {}
    for row in payload.get('artifacts', []):
        relative = row.get('relative_path')
        if not relative and row.get('mirror'):
            mirror_path = Path(row['mirror'])
            try:
                relative = mirror_path.relative_to(declared_root).as_posix()
            except ValueError:
                relative = mirror_path.name
        if relative:
            rows[Path(relative).as_posix()] = row
    return manifest_path, rows


def _restore_verified_revision_artifact_05(relative, destination):
    import shutil

    relative = Path(relative).as_posix()
    destination = Path(destination)
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    row = rows.get(relative)
    source = MIRROR_REVISION_ROOT / relative
    if row is None or not source.exists() or source.stat().st_size == 0:
        return 'unverified_active' if destination.exists() and destination.stat().st_size > 0 else 'unavailable'
    expected_size = int(row.get('size_bytes', -1))
    expected_sha = str(row.get('sha256') or '')
    if expected_size >= 0 and source.stat().st_size != expected_size:
        raise RuntimeError(f'Mirror size mismatch for artifact: {relative}')
    if len(expected_sha) != 64 or _sha256_local(source) != expected_sha:
        raise RuntimeError(f'Mirror checksum mismatch for artifact: {relative}')
    if destination.exists() and destination.stat().st_size > 0 and _sha256_local(destination) == expected_sha:
        return 'reused'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    if _sha256_local(destination) != expected_sha:
        raise RuntimeError(f'Checksum mismatch after restoring artifact: {relative}')
    return 'restored'


def restore_required_revision_artifacts(required_rel_paths):
    destination_root = REPO_DIR / 'reports' / 'revision'
    restored, reused, unresolved = [], [], []
    for rel in required_rel_paths:
        rel = Path(rel).as_posix()
        status = _restore_verified_revision_artifact_05(rel, destination_root / rel)
        if status == 'restored':
            restored.append(rel)
        elif status == 'reused':
            reused.append(rel)
        elif status == 'unverified_active':
            raise RuntimeError(f'Active required artifact is absent from the canonical mirror manifest: {rel}')
        else:
            unresolved.append(rel)
    print(
        'Required revision artifact restore: '
        f'restored={len(restored)} reused={len(reused)} unresolved={len(unresolved)}'
    )
    for rel in restored:
        print(' - restored', rel)
    for rel in unresolved:
        print(' - unresolved (missing from verified mirror manifest)', rel)


RUN_FULL_MIRROR_RESTORE_05 = os.environ.get('ECG_RAMBA_FULL_MIRROR_RESTORE_05', '0') == '1'
if MIRROR_REVISION_ROOT.exists():
    if RUN_FULL_MIRROR_RESTORE_05:
        restore_result = _run_setup(
            f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
            cwd=REPO_DIR,
            check=False,
        )
        if restore_result.returncode:
            print('WARNING: full checksum-verified mirror restore failed; trying selective restore for required inputs only.')
    else:
        print('Skipping full mirror restore in Notebook 05 setup. Set ECG_RAMBA_FULL_MIRROR_RESTORE_05=1 only if a full local mirror copy is required.')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)
else:
    raise FileNotFoundError(f'Canonical artifact mirror not found: {MIRROR_REVISION_ROOT}')

LEARNED_COMPARATOR_CHECKPOINT_ARTIFACTS_FOR_05 = [
    *(f'experimental/resnet1d_cnn_checkpoints/fold{i}_resnet1d_cnn_final.pt' for i in range(1, 6)),
    *(f'experimental/raw_mamba_checkpoints/fold{i}_raw_mamba_final_ema.pt' for i in range(1, 6)),
    *(f'experimental/transformer_ecg_checkpoints/fold{i}_transformer_ecg_final.pt' for i in range(1, 6)),
]


def canonical_learned_comparator_checkpoint_status_05(required_rel_paths=None):
    required_rel_paths = list(required_rel_paths or LEARNED_COMPARATOR_CHECKPOINT_ARTIFACTS_FOR_05)
    ready, unresolved = [], []
    for rel in required_rel_paths:
        path = MIRROR_REVISION_ROOT / Path(rel)
        if path.exists() and path.stat().st_size > 0:
            ready.append(path.as_posix())
        else:
            unresolved.append(path.as_posix())
    print(f'Canonical learned-comparator checkpoints: ready={len(ready)} unresolved={len(unresolved)}')
    for item in unresolved:
        print(' - unresolved', item)
    return {'ready': ready, 'unresolved': unresolved}


checkpoint_restore_report_05 = canonical_learned_comparator_checkpoint_status_05()

required_code = [
    'scripts/revision/09_hrv_domain_analysis.py',
    'scripts/revision/12_robustness_stress.py',
    'scripts/revision/14_resnet1d_cnn_baseline.py',
    'scripts/revision/16_raw_mamba_baseline.py',
    'scripts/revision/21_robustness_multicomparator.py',
    'scripts/revision/23_generate_comparator_stress_predictions.py',
    'scripts/revision/24_transformer_ecg_baseline.py',
    'scripts/revision/robustness_profile_audit.py',
    'notebooks/05_hrv_domain_and_robustness.ipynb',
    'docs/revision_plan/experiment_registry.csv',
    'docs/revision_plan/task_board.csv',
]
missing_code = []
for item in required_code:
    file_path = REPO_DIR / item
    status = 'FOUND' if file_path.exists() else 'MISSING'
    size = file_path.stat().st_size if file_path.exists() else None
    print(f'{item}: {status} size={size}')
    if not file_path.exists():
        missing_code.append(item)
if missing_code:
    raise FileNotFoundError('Missing required code files after setup: ' + '; '.join(missing_code))

compatibility_tokens_05 = {
    'scripts/revision/12_robustness_stress.py': [
        'nominal_95_percentile_paired_record_bootstrap_unadjusted',
        'full_nominal_95ci_less_degraded',
        'nominal_95ci_inconclusive_stressed_difference',
    ],
    'scripts/revision/artifact_mirror.py': ['--verify-existing', 'discovered_unmanifested_count', '--source-conflict-policy'],
    'scripts/revision/23_generate_comparator_stress_predictions.py': [
        '--transformer-checkpoint-dir',
        'transformer_helpers',
        'fold{fold}_transformer_ecg_final.pt',
        'robustness_{stem}_{stress}_predictions.npz',
        '--finalize-manifest-only',
        'canonical_contract',
        'validate_checkpoint_set',
    ],
    'scripts/revision/21_robustness_multicomparator.py': [
        'Transformer ECG',
        'transformer_ecg_oof_predictions.npz',
        'robustness_minirocket_clean_ref_predictions.npz',
        'validate_stress_provenance',
        'checkpoint_contract',
        'source_pairwise_sha256',
        'output_profile_name',
        'OOF_RUN_MANIFEST',
        'robustness_transformer_ecg_{stress}_predictions.npz',
        '--metrics',
        '--bootstrap-jobs',
        'paired_record_resample_presorted_rank_sparse_ece_weighted_counts_v2',
        'weighted_resample_metric',
        'rank_context',
        'ece_context',
        'bootstrap_record_counts',
        'nominal_95_percentile_paired_record_bootstrap_unadjusted',
        'fixed_trained_folds_and_checkpoints_not_retrained_within_bootstrap',
        'METRIC_CACHE_SCHEMA_VERSION = ROBUSTNESS_METRIC_CACHE_SCHEMA_VERSION',
        'rank_calibration_omit_single_resampled_class_f1_keeps_all_labels_zero_division_zero',
        'load_bootstrap_independence_contract',
        'expected_stress_spec',
        'load_validated_clean_artifact',
        'load_validated_stress_artifact',
        'filter_metric_specs',
        'clean prediction checkpoint SHA contract differs from its baseline manifest',
    ],
}


def transformer_compatibility_failures_05():
    failures = []
    for rel_path, tokens in compatibility_tokens_05.items():
        source_path = REPO_DIR / rel_path
        source_text = source_path.read_text(encoding='utf-8', errors='replace') if source_path.exists() else ''
        missing_tokens = [token for token in tokens if token not in source_text]
        if missing_tokens:
            failures.append(f'{rel_path}: missing tokens {missing_tokens}')
    return failures


def refresh_transformer_aware_scripts_05():
    import urllib.request

    rel_paths = list(compatibility_tokens_05)
    print('Notebook 05 robustness script preflight failed; attempting targeted refresh from origin/GitHub raw.')
    _run_setup(f'git fetch origin {BRANCH}', cwd=REPO_DIR, check=False)
    checkout_command = 'git checkout origin/{branch} -- {paths}'.format(
        branch=BRANCH,
        paths=' '.join(rel_paths),
    )
    checkout_result = _run_setup(checkout_command, cwd=REPO_DIR, check=False)
    if checkout_result.returncode == 0:
        print('Restored Transformer-aware scripts from git origin target.')
    else:
        print('Targeted git checkout failed; falling back to GitHub raw download for required scripts.')

    remaining = transformer_compatibility_failures_05()
    if remaining:
        base_url = f'https://raw.githubusercontent.com/BrianNguyen29/ECG-RAMBA/{BRANCH}/'
        for rel_path in rel_paths:
            destination = REPO_DIR / rel_path
            url = base_url + rel_path
            print('Fetching raw file:', url)
            try:
                text = urllib.request.urlopen(url, timeout=60).read().decode('utf-8')
            except Exception as exc:
                print(f'WARNING: raw fetch failed for {rel_path}: {exc}')
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_text(text, encoding='utf-8')
        remaining = transformer_compatibility_failures_05()
    return remaining


compatibility_failures_05 = transformer_compatibility_failures_05()
if compatibility_failures_05:
    compatibility_failures_05 = refresh_transformer_aware_scripts_05()
if compatibility_failures_05:
    raise RuntimeError(
        'Notebook 05 requires the current robustness scripts, and automatic targeted refresh failed. '
        'Rerun this Notebook 05 Setup cell, or run `git pull origin main` in /content/ECG-RAMBA, then rerun Notebook 05. '
        + ' ; '.join(compatibility_failures_05)
    )
print('Notebook 05 robustness script preflight: OK')

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
    'model_dir_drive': DRIVE_ROOT / 'model',
    'canonical_checkpoint_root': CANONICAL_CHECKPOINT_ROOT,
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, cache_path in cache_status.items():
    if cache_path.is_dir():
        npz_count = len(list(cache_path.glob('*.npz')))
        pt_count = len(list(cache_path.glob('*.pt')))
        print(f'  {name}: exists=True npz_count={npz_count} pt_count={pt_count} path={cache_path}')
    else:
        print(f'  {name}: exists={cache_path.exists()} size={cache_path.stat().st_size if cache_path.exists() else None} path={cache_path}')

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    print(f'$ {cmd}', flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)

    from contextlib import ExitStack
    with ExitStack() as stack:
        log_file = stack.enter_context(log_path.open('a', encoding='utf-8'))
        durable_log_file = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
            durable_log_file.write(line)
            durable_log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    print(f'Durable command log: {durable_log_path}')
    if check and return_code:
        log_lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(log_lines))} log lines:')
        for line in log_lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    script = Path(script_path)
    if script.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')

print('Setup ready. Continue with Scope And Contracts cell.')

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    return_code = -1
    caught = None
    completed = None
    process = None
    try:
        process = _forensic_subprocess.Popen(
            cmd,
            shell=isinstance(cmd, str),
            cwd=str(run_cwd),
            stdout=_forensic_subprocess.PIPE,
            stderr=_forensic_subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        with history_log.open('a', encoding='utf-8') as local_handle:
            canonical_handle = (
                canonical_history.open('a', encoding='utf-8')
                if canonical_history is not None
                else None
            )
            try:
                for line in process.stdout or ():
                    print(line, end='', flush=True)
                    local_handle.write(line)
                    local_handle.flush()
                    if canonical_handle is not None:
                        canonical_handle.write(line)
                        canonical_handle.flush()
                return_code = int(process.wait())
                local_handle.flush()
                _forensic_os.fsync(local_handle.fileno())
                if canonical_handle is not None:
                    canonical_handle.flush()
                    _forensic_os.fsync(canonical_handle.fileno())
            finally:
                if canonical_handle is not None:
                    canonical_handle.close()
        completed = _forensic_subprocess.CompletedProcess(cmd, return_code)
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except Exception:
                process.kill()
                process.wait()
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed


## Scope And Contracts

This notebook is direct-run capable: it mounts Drive, refreshes the GitHub checkout, installs base packages, executes HRV-only/domain evidence, Full-vs-MiniRocket perturbation robustness, optional comparator stressed-prediction generation for ResNet1D/CNN, Raw Mamba, and Transformer ECG, and a low-memory multi-comparator robustness ledger. HRV/domain cells do not load neural checkpoints. Robustness inference is deliberately gated behind explicit flags and requires cached predictions or saved comparator checkpoints.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

RUN_HRV_DOMAIN_ANALYSIS = True
FORCE_RERUN_HRV_DOMAIN_ANALYSIS = False
RUN_ROBUSTNESS_STRESS = True
FORCE_RERUN_ROBUSTNESS_STRESS = False

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
oof_prediction_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
baseline_summary_path = revision_root / 'metrics' / 'baseline_summary.csv'
component_summary_path = revision_root / 'metrics' / 'component_check_summary.json'
audit_path = revision_root / 'audit_protocol.json'
a0_status_path = revision_root / 'a0_resolution_status.json'
hrv_schema_path = revision_root / 'hrv36_schema.csv'

required_inputs = {
    'oof_final_ema_freeze_manifest': freeze_path,
    'oof_final_ema_predictions': oof_prediction_path,
    'calibration_ci': calibration_path,
    'baseline_summary': baseline_summary_path,
    'component_check_summary': component_summary_path,
    'audit_protocol': audit_path,
    'a0_resolution_status': a0_status_path,
    'hrv36_schema': hrv_schema_path,
}
for name, path in required_inputs.items():
    print(f'{name:26s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' in globals():
    print('Required inputs missing before contract validation; retrying selective artifact restore...')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' not in globals():
    raise RuntimeError('Run Notebook 05 from Setup; unverified direct Drive copies are disabled.')
if missing:
    raise FileNotFoundError(
        'Notebook 01/02/03/04 artifacts are required before notebook 05: ' + '; '.join(missing) +
        '. Run Notebook 05 from the Setup cell and verify the canonical mirror at '
        '/content/drive/MyDrive/ECG-Ramba/revision_artifacts/reports/revision.'
    )

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before HRV/robustness review.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 05 requires checkpoint_kind=final_ema in the freeze manifest.')
frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
relative_oof_predictions = oof_prediction_path.as_posix()
if relative_oof_predictions not in frozen_artifacts:
    raise ValueError(f'Freeze manifest does not include {relative_oof_predictions}')
oof_prediction_sha = sha256_file(oof_prediction_path)
if oof_prediction_sha != frozen_artifacts[relative_oof_predictions]['sha256']:
    raise RuntimeError('OOF prediction SHA256 changed after final_ema freeze.')

baseline_df_for_contract = pd.read_csv(baseline_summary_path)
full_rows = baseline_df_for_contract[baseline_df_for_contract['baseline_name'] == 'Full ECG-RAMBA frozen OOF']
if full_rows.empty:
    raise RuntimeError('baseline_summary.csv lacks the Full ECG-RAMBA frozen OOF row from Notebook 04.')
if set(full_rows['evidence_path'].astype(str)) != {str(calibration_path)}:
    raise RuntimeError('baseline_summary.csv full-model row is not tied to the canonical final_ema calibration artifact.')
component_summary = json.loads(component_summary_path.read_text(encoding='utf-8'))
if component_summary.get('source_freeze_manifest') != str(freeze_path):
    raise RuntimeError('component_check_summary.json is not tied to the canonical final_ema freeze manifest.')
if component_summary.get('source_calibration') != str(calibration_path):
    raise RuntimeError('component_check_summary.json is not tied to the canonical final_ema calibration artifact.')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'freeze_checkpoint_kind': freeze.get('checkpoint_kind'),
    'oof_predictions_sha256': oof_prediction_sha,
    'allowed_execution': {
        'hrv_only_feature_baseline_training': RUN_HRV_DOMAIN_ANALYSIS,
        'hrv_domain_classifier_training': RUN_HRV_DOMAIN_ANALYSIS,
        'force_rerun_hrv_domain_analysis': FORCE_RERUN_HRV_DOMAIN_ANALYSIS,
        'robustness_stress_runner': RUN_ROBUSTNESS_STRESS,
        'force_rerun_robustness_stress': FORCE_RERUN_ROBUSTNESS_STRESS,
        'full_model_training': False,
        'full_model_inference': RUN_ROBUSTNESS_STRESS,
        'signal_perturbation': RUN_ROBUSTNESS_STRESS,
    },
}
save_json(revision_root / 'manifests' / 'hrv_robustness_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'hrv_robustness_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 05 to HRV-domain and robustness tasks in the tracked revision plan.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_HRV', 'EXP_ROBUST'])]
relevant_claims = claims[claims['claim_id'].isin(['C03'])]
relevant_tasks = tasks[tasks['id'].isin(['A6', 'B2'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json')


## HRV Domain Evidence

This cell runs the implemented HRV-only baseline and HRV-domain classifier. The HRV-only baseline uses HRV36 features with the same frozen Chapman OOF folds. The domain classifier uses bounded HRV36 feature caches only; it does not open raw signals or run the neural model.


In [ ]:
hrv_schema = pd.read_csv(hrv_schema_path)
print('HRV schema rows:', len(hrv_schema))
display(hrv_schema.head(10))

with np.load(oof_prediction_path, allow_pickle=False) as oof_npz:
    if 'dataset_record_order_fingerprint' not in oof_npz.files:
        raise RuntimeError('Canonical final_ema OOF predictions must contain dataset_record_order_fingerprint.')
    dataset_record_order_fingerprint = str(oof_npz['dataset_record_order_fingerprint'].item())

chapman_hrv_cache = DRIVE_ROOT / f'hrv36_N44186_C12_L5000_R{dataset_record_order_fingerprint}.npz'
if not chapman_hrv_cache.exists():
    available_caches = sorted(path.name for path in DRIVE_ROOT.glob('hrv36_N44186_C12_L5000_R*.npz'))
    raise FileNotFoundError(
        'Missing Chapman HRV cache that matches the final_ema OOF record-order fingerprint: '
        f'{chapman_hrv_cache}. Available fingerprinted caches: {available_caches}. '
        'Run Notebook 02a/train cache generation for the same Chapman record order; do not use a mismatched HRV cache.'
    )
chapman_hrv_cache_sha = sha256_file(chapman_hrv_cache)
print('OOF dataset record-order fingerprint:', dataset_record_order_fingerprint)
print('Using matching Chapman HRV cache:', chapman_hrv_cache)
print('Chapman HRV cache sha256:', chapman_hrv_cache_sha)

hrv_command = (
    'python -u scripts/revision/09_hrv_domain_analysis.py '
    f'--oof-predictions {oof_prediction_path} '
    f'--freeze-manifest {freeze_path} --expected-checkpoint-kind final_ema '
    f'--chapman-hrv-cache "{chapman_hrv_cache}" '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --domain-max-per-domain 2000'
)
hrv_prediction_path = revision_root / 'predictions' / 'hrv_only_oof_predictions.npz'
hrv_reuse_attestation_path = revision_root / 'metrics' / 'hrv_only_oof_reuse_attestation.json'
hrv_required_outputs = [
    hrv_prediction_path,
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
]
hrv_optional_outputs = [
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_fold_summary.csv',
]


def _hrv_restore_roots():
    candidates = []
    if 'MIRROR_REVISION_ROOT' in globals():
        candidates.append(Path(MIRROR_REVISION_ROOT))
    candidates.append(DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
    roots = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        key = candidate.as_posix()
        if key not in seen and candidate.exists():
            roots.append(candidate)
            seen.add(key)
    return roots


def _restore_hrv_artifacts(paths):
    restored, unresolved, reused = [], [], []
    for destination in paths:
        destination = Path(destination)
        relative = destination.relative_to(revision_root).as_posix()
        status = _restore_verified_revision_artifact_05(relative, destination)
        if status == 'restored':
            restored.append(relative)
        elif status == 'reused':
            reused.append(relative)
        elif status == 'unverified_active':
            raise RuntimeError(f'Active artifact is absent from the canonical mirror manifest: {relative}')
        else:
            unresolved.append(relative)
    if restored or unresolved:
        print(f'HRV/domain artifact restore: restored={len(restored)} reused={len(reused)} unresolved={len(unresolved)}')
    return {'restored': restored, 'reused': reused, 'unresolved': unresolved}


_restore_hrv_artifacts(hrv_required_outputs + hrv_optional_outputs + [hrv_reuse_attestation_path])


def _oof_label_fold_contract(path):
    import importlib

    hrv_runner = importlib.import_module('scripts.revision.09_hrv_domain_analysis')
    with np.load(path, allow_pickle=False) as payload:
        required = {'y_true', 'fold_id', 'record_id', 'class_names'}
        missing = required - set(payload.files)
        if missing:
            raise KeyError(f'{path} lacks semantic OOF fields: {sorted(missing)}')
        return hrv_runner.oof_label_fold_contract_sha256(
            y_true=np.asarray(payload['y_true'], dtype=np.float32),
            fold_id=np.asarray(payload['fold_id'], dtype=np.int16),
            record_id=np.asarray(payload['record_id'], dtype=np.int64),
            class_names=np.asarray(payload['class_names']).astype(str).tolist(),
        )


def _inspect_hrv_reuse_contract():
    required_present = all(output.exists() and output.stat().st_size > 0 for output in hrv_required_outputs)
    audit = {
        'required_artifacts_present': required_present,
        'current_oof_sha256': oof_prediction_sha,
        'current_freeze_sha256': sha256_file(freeze_path),
        'issues': [],
    }
    if not required_present:
        audit['issues'].append('required_hrv_artifacts_missing')
        audit['semantic_contract_match'] = False
        return audit
    try:
        baseline_summary = json.loads((revision_root / 'metrics' / 'hrv_only_baseline_summary.json').read_text(encoding='utf-8'))
        hrv_runner_path = Path('scripts/revision/09_hrv_domain_analysis.py')
        hrv_manifest_path = revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json'
        hrv_manifest = json.loads(hrv_manifest_path.read_text(encoding='utf-8'))
        active_freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
        active_group = (active_freeze.get('group_contract') or {}).get('sidecar') or {}
        active_group_sha = str(active_group.get('sha256') or '')
        bootstrap_contract = baseline_summary.get('bootstrap_contract') or {}
        audit.update({
            'runner_sha256': baseline_summary.get('runner_sha256'),
            'group_sidecar_sha256': bootstrap_contract.get('group_sidecar_sha256'),
            'bootstrap_unit': bootstrap_contract.get('unit'),
        })
        if int(baseline_summary.get('schema_version', 0)) < 2:
            audit['issues'].append('hrv_runner_schema_stale')
        if baseline_summary.get('runner_sha256') != sha256_file(hrv_runner_path):
            audit['issues'].append('hrv_runner_sha_mismatch')
        if int(bootstrap_contract.get('n_boot', -1)) != 1000:
            audit['issues'].append('hrv_bootstrap_count_mismatch')
        if bootstrap_contract.get('method') != 'percentile_cluster_bootstrap':
            audit['issues'].append('hrv_bootstrap_method_mismatch')
        if bootstrap_contract.get('unit') != 'authenticated_source_patient_record':
            audit['issues'].append('hrv_bootstrap_unit_mismatch')
        if bootstrap_contract.get('one_record_per_group') is not True:
            audit['issues'].append('hrv_group_independence_not_verified')
        if not active_group_sha or bootstrap_contract.get('group_sidecar_sha256') != active_group_sha:
            audit['issues'].append('hrv_group_sidecar_sha_mismatch')
        manifest_contract = hrv_manifest.get('canonical_contract') or {}
        if hrv_manifest.get('runner_sha256') != sha256_file(hrv_runner_path):
            audit['issues'].append('hrv_manifest_runner_sha_mismatch')
        if manifest_contract.get('group_sidecar_sha256') != active_group_sha:
            audit['issues'].append('hrv_manifest_group_sidecar_sha_mismatch')
        load_info = baseline_summary.get('load_info', {})
        freeze_contract = load_info.get('freeze_contract', {}) or {}
        current_semantic_sha = _oof_label_fold_contract(oof_prediction_path)
        cached_semantic_sha = _oof_label_fold_contract(hrv_prediction_path)
        audit.update({
            'current_oof_semantic_sha256': current_semantic_sha,
            'cached_hrv_semantic_sha256': cached_semantic_sha,
            'semantic_contract_match': current_semantic_sha == cached_semantic_sha,
            'summary_oof_sha256': load_info.get('oof_predictions_sha256'),
            'summary_freeze_sha256': freeze_contract.get('freeze_manifest_sha256'),
            'exact_oof_sha_match': load_info.get('oof_predictions_sha256') == oof_prediction_sha,
            'exact_freeze_sha_match': freeze_contract.get('freeze_manifest_sha256') == sha256_file(freeze_path),
            'checkpoint_kind': freeze_contract.get('checkpoint_kind'),
            'chapman_hrv_cache_sha256': load_info.get('chapman_hrv_cache_sha256'),
        })
        if not audit['semantic_contract_match']:
            audit['issues'].append('oof_label_fold_semantic_contract_mismatch')
        if freeze_contract.get('checkpoint_kind') != 'final_ema':
            audit['issues'].append('checkpoint_kind_not_final_ema')
        if load_info.get('chapman_hrv_cache_kind') != 'record_fingerprinted':
            audit['issues'].append('hrv_cache_not_record_fingerprinted')
        if load_info.get('chapman_hrv_cache_sha256') != chapman_hrv_cache_sha:
            audit['issues'].append('hrv_cache_sha_mismatch')
        if dataset_record_order_fingerprint not in Path(load_info.get('chapman_hrv_cache', '')).name:
            audit['issues'].append('hrv_cache_record_order_fingerprint_mismatch')
    except Exception as exc:
        audit['semantic_contract_match'] = False
        audit['issues'].append(f'{type(exc).__name__}: {exc}')
    return audit


hrv_reuse_contract = _inspect_hrv_reuse_contract()
hrv_artifacts_ready = not hrv_reuse_contract['issues']
if hrv_reuse_contract['issues']:
    print('HRV/domain cache is not reusable:', hrv_reuse_contract['issues'])
if RUN_HRV_DOMAIN_ANALYSIS and (FORCE_RERUN_HRV_DOMAIN_ANALYSIS or not hrv_artifacts_ready):
    run(hrv_command, log_path='reports/revision/logs/hrv_domain_analysis.log')
    hrv_reuse_contract = _inspect_hrv_reuse_contract()
    hrv_artifacts_ready = not hrv_reuse_contract['issues']
elif hrv_artifacts_ready:
    print('Reusing semantically verified HRV/domain artifacts. Set FORCE_RERUN_HRV_DOMAIN_ANALYSIS=True to recompute.')
else:
    raise RuntimeError('RUN_HRV_DOMAIN_ANALYSIS is False and required HRV/domain artifacts are missing; HRV evidence will remain blocked.')

if not hrv_artifacts_ready:
    raise RuntimeError('HRV/domain artifacts failed semantic OOF reuse validation: ' + '; '.join(hrv_reuse_contract['issues']))
for output in hrv_required_outputs:
    print(output, 'required exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)
missing_hrv = [str(output) for output in hrv_required_outputs if not output.exists()]
if missing_hrv:
    raise FileNotFoundError('Missing required HRV analysis outputs: ' + '; '.join(missing_hrv))
for output in hrv_optional_outputs:
    print(output, 'optional exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)

hrv_df = pd.read_csv(revision_root / 'metrics' / 'hrv_domain_summary.csv')
hrv_baseline_summary = json.loads((revision_root / 'metrics' / 'hrv_only_baseline_summary.json').read_text(encoding='utf-8'))
hrv_domain_summary = json.loads((revision_root / 'metrics' / 'hrv_domain_classifier_summary.json').read_text(encoding='utf-8'))
hrv_load_info = hrv_baseline_summary.get('load_info', {})
hrv_freeze_contract = hrv_load_info.get('freeze_contract', {}) or {}
exact_oof_sha_match = hrv_load_info.get('oof_predictions_sha256') == oof_prediction_sha
if not hrv_reuse_contract.get('semantic_contract_match'):
    raise RuntimeError('HRV-only OOF labels/fold assignments do not match the canonical final_ema OOF semantic contract.')
if not exact_oof_sha_match:
    print(
        'HRV-only summary OOF file SHA differs from the current rewritten OOF NPZ, '
        'but y_true/fold_id/record_id/class_names match exactly by semantic SHA; reusing HRV artifact.'
    )
if hrv_load_info.get('chapman_hrv_cache_kind') != 'record_fingerprinted':
    raise RuntimeError('HRV-only summary did not use a record-fingerprinted Chapman HRV cache.')
if hrv_load_info.get('chapman_hrv_cache_sha256') != chapman_hrv_cache_sha:
    raise RuntimeError('HRV-only summary cache SHA does not match the selected fingerprinted HRV cache.')
if dataset_record_order_fingerprint not in Path(hrv_load_info.get('chapman_hrv_cache', '')).name:
    raise RuntimeError('HRV-only summary cache does not match the canonical final_ema record-order fingerprint.')
if hrv_freeze_contract.get('checkpoint_kind') != 'final_ema':
    raise RuntimeError('HRV-only summary is not tied to final_ema freeze contract.')

hrv_reuse_attestation = {
    'status': 'verified',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'current_oof_sha256': oof_prediction_sha,
    'summary_oof_sha256': hrv_load_info.get('oof_predictions_sha256'),
    'exact_oof_sha_match': exact_oof_sha_match,
    'current_freeze_sha256': sha256_file(freeze_path),
    'summary_freeze_sha256': hrv_freeze_contract.get('freeze_manifest_sha256'),
    'exact_freeze_sha_match': hrv_freeze_contract.get('freeze_manifest_sha256') == sha256_file(freeze_path),
    'current_oof_semantic_sha256': hrv_reuse_contract.get('current_oof_semantic_sha256'),
    'cached_hrv_semantic_sha256': hrv_reuse_contract.get('cached_hrv_semantic_sha256'),
    'semantic_contract_match': hrv_reuse_contract.get('semantic_contract_match') is True,
    'semantic_fields': ['y_true', 'fold_id', 'record_id', 'class_names'],
    'hrv_prediction_sha256': sha256_file(hrv_prediction_path),
    'chapman_hrv_cache_sha256': chapman_hrv_cache_sha,
    'issues': hrv_reuse_contract.get('issues', []),
}
save_json(hrv_reuse_attestation_path, hrv_reuse_attestation)
print('Wrote HRV OOF reuse attestation:', hrv_reuse_attestation_path)
print(json.dumps(hrv_reuse_attestation, indent=2, sort_keys=True))

# Publish immediately so a later Notebook 04 rerun can restore the canonical HRV artifacts
# even if the Colab runtime disconnects before the final inventory cell.
mirror_root = globals().get('MIRROR_REVISION_ROOT', DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_domain_immediate_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_domain_immediate_mirror_publish.log',
)

print('HRV-only metrics:', json.dumps(hrv_baseline_summary['metrics'], indent=2, sort_keys=True))
print('HRV domain status:', hrv_domain_summary.get('status', 'complete'))
print('HRV domain metrics:', json.dumps(hrv_domain_summary.get('metrics') or {}, indent=2, sort_keys=True))
print('HRV domain interpretation:', hrv_domain_summary.get('interpretation'))
if hrv_domain_summary.get('blocker'):
    print('HRV domain blocker:', hrv_domain_summary.get('blocker'))
display(hrv_df)


## Robustness Stress-Test Ledger

The revised plan requires perturbation tests. This cell is configured as a final aggregation pass: it checks that all six per-stress Full/MiniRocket prediction artifacts exist, restores the Drive mirror once if needed, then runs the robustness script with `--reuse-existing` and Drive-backed metric-level resume caches to regenerate the combined tables, bootstrap CIs, and manifest. If Colab disconnects mid-bootstrap, rerun this cell; completed stress/metric caches are reused instead of recomputed. If a prediction artifact is missing, run that stress individually first and publish the mirror before returning to the full list.


In [ ]:
ROBUSTNESS_STRESS_TESTS = 'snr20db,snr10db,snr5db,random_3_lead_dropout,precordial_dropout,resample_250hz'
ROBUSTNESS_LIMIT_RECORDS = 0
ROBUSTNESS_N_BOOT = 1000
ROBUSTNESS_BATCH_SIZE = int(os.environ.get('ECG_RAMBA_ROBUSTNESS_BATCH_SIZE', '64'))
ROBUSTNESS_NUM_WORKERS = 0
ROBUSTNESS_MINIROCKET_FEATURE_DEVICE = 'cpu'
ROBUSTNESS_ALLOW_TF32 = True
ROBUSTNESS_FINAL_AGGREGATION_PASS = True
ROBUSTNESS_REQUIRE_EXISTING_STRESS_PREDICTIONS = True
ROBUSTNESS_METRIC_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'robustness_metric_cache'

robustness_outputs = {
    'summary_csv': revision_root / 'metrics' / 'robustness_summary.csv',
    'table_csv': revision_root / 'tables' / 'table_robustness.csv',
    'comparison_json': revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json',
    'bootstrap_samples': revision_root / 'metrics' / 'robustness_full_vs_minirocket_bootstrap_samples.csv',
    'manifest': revision_root / 'manifests' / 'robustness_stress_manifest.json',
    'stress_metadata': revision_root / 'tables' / 'table_robustness_stress_metadata.csv',
    'artifacts': revision_root / 'tables' / 'table_robustness_artifacts.csv',
}


def _robustness_restore_roots():
    candidates = []
    if 'MIRROR_REVISION_ROOT' in globals():
        candidates.append(Path(MIRROR_REVISION_ROOT))
    candidates.append(DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
    roots = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        key = candidate.as_posix()
        if key not in seen and candidate.exists():
            roots.append(candidate)
            seen.add(key)
    return roots


def _restore_revision_artifacts_from_drive(paths, label):
    restored, reused, unresolved = [], [], []
    for destination in paths:
        destination = Path(destination)
        relative = destination.relative_to(revision_root).as_posix()
        status = _restore_verified_revision_artifact_05(relative, destination)
        if status == 'restored':
            restored.append(relative)
        elif status == 'reused':
            reused.append(relative)
        elif status == 'unverified_active':
            raise RuntimeError(f'Active artifact is absent from the canonical mirror manifest: {relative}')
        else:
            unresolved.append(relative)
    if restored or unresolved:
        print(f'{label} restore: restored={len(restored)} reused={len(reused)} unresolved={len(unresolved)}')
    return {'restored': restored, 'reused': reused, 'unresolved': unresolved}


_restore_revision_artifacts_from_drive(list(robustness_outputs.values()), 'Robustness aggregate outputs')
robustness_outputs_ready = all(path.exists() and path.stat().st_size > 0 for path in robustness_outputs.values())

robustness_stress_list = [s.strip() for s in ROBUSTNESS_STRESS_TESTS.split(',') if s.strip()]
expected_full_robustness_stresses = [
    'snr20db',
    'snr10db',
    'snr5db',
    'random_3_lead_dropout',
    'precordial_dropout',
    'resample_250hz',
]
expected_robustness_metrics = {'pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro'}
robustness_reference_outputs = [
    revision_root / 'predictions' / 'robustness_minirocket_clean_ref_predictions.npz',
    revision_root / 'manifests' / 'robustness_minirocket_heads_manifest.json',
]
robustness_prediction_outputs = [
    revision_root / 'predictions' / f'robustness_{model_label}_{stress_name}_predictions.npz'
    for stress_name in robustness_stress_list
    for model_label in ['full', 'minirocket']
]

if ROBUSTNESS_FINAL_AGGREGATION_PASS and robustness_stress_list != expected_full_robustness_stresses:
    raise ValueError(
        'Final aggregation pass must use the canonical full stress list. '
        f'Got: {robustness_stress_list}'
    )


def robustness_prediction_status(stress_names):
    rows = []
    for stress_name in stress_names:
        for model_label in ['full', 'minirocket']:
            path = revision_root / 'predictions' / f'robustness_{model_label}_{stress_name}_predictions.npz'
            rows.append({
                'stress_test': stress_name,
                'model': model_label,
                'path': path.as_posix(),
                'exists': path.exists(),
                'size_bytes': path.stat().st_size if path.exists() else None,
            })
    return pd.DataFrame(rows)


def missing_robustness_prediction_paths(status_df):
    return status_df.loc[~status_df['exists'], 'path'].astype(str).tolist()


robustness_predictions_complete = False
missing_robustness_stresses = []
missing_robustness_references = []
needs_robustness_regeneration = RUN_ROBUSTNESS_STRESS and (FORCE_RERUN_ROBUSTNESS_STRESS or not robustness_outputs_ready)

if RUN_ROBUSTNESS_STRESS and robustness_outputs_ready and not FORCE_RERUN_ROBUSTNESS_STRESS:
    print('Reusing existing complete robustness summary artifacts; skipping per-stress prediction cache preflight.')
    print('Set FORCE_RERUN_ROBUSTNESS_STRESS=True only when you intentionally want to regenerate robustness outputs.')
elif needs_robustness_regeneration and ROBUSTNESS_FINAL_AGGREGATION_PASS and ROBUSTNESS_REQUIRE_EXISTING_STRESS_PREDICTIONS:
    print('Final aggregation preflight: checking per-stress prediction artifacts before running robustness summary generation.')
    _restore_revision_artifacts_from_drive(robustness_prediction_outputs + robustness_reference_outputs, 'Robustness prediction/reference artifacts')
    prediction_status = robustness_prediction_status(robustness_stress_list)
    display(prediction_status)
    reference_status = pd.DataFrame([
        {
            'artifact': path.name,
            'path': path.as_posix(),
            'exists': path.exists(),
            'size_bytes': path.stat().st_size if path.exists() else None,
        }
        for path in robustness_reference_outputs
    ])
    display(reference_status)
    missing_predictions = missing_robustness_prediction_paths(prediction_status)
    missing_references = reference_status.loc[~reference_status['exists'], 'path'].astype(str).tolist()
    missing_predictions.extend(missing_references)
    if missing_predictions and 'MIRROR_REVISION_ROOT' in globals() and MIRROR_REVISION_ROOT.exists():
        print('Missing prediction artifacts after setup; retrying canonical targeted restore only.')
        _restore_revision_artifacts_from_drive(
            robustness_prediction_outputs + robustness_reference_outputs,
            'Robustness prediction/reference artifacts retry',
        )
        prediction_status = robustness_prediction_status(robustness_stress_list)
        display(prediction_status)
        reference_status = pd.DataFrame([
            {
                'artifact': path.name,
                'path': path.as_posix(),
                'exists': path.exists(),
                'size_bytes': path.stat().st_size if path.exists() else None,
            }
            for path in robustness_reference_outputs
        ])
        display(reference_status)
        missing_predictions = missing_robustness_prediction_paths(prediction_status)
        missing_predictions.extend(reference_status.loc[~reference_status['exists'], 'path'].astype(str).tolist())
    missing_robustness_stresses = [
        stress_name
        for stress_name in robustness_stress_list
        if any(
            not path.exists() or path.stat().st_size == 0
            for path in [
                revision_root / 'predictions' / f'robustness_full_{stress_name}_predictions.npz',
                revision_root / 'predictions' / f'robustness_minirocket_{stress_name}_predictions.npz',
            ]
        )
    ]
    missing_robustness_references = [
        path for path in robustness_reference_outputs if not path.exists() or path.stat().st_size == 0
    ]
    if FORCE_RERUN_ROBUSTNESS_STRESS:
        missing_robustness_stresses = list(robustness_stress_list)
    robustness_predictions_complete = not missing_robustness_stresses and not missing_robustness_references
    if robustness_predictions_complete:
        print('All per-stress prediction artifacts are present. The runner will reuse predictions and regenerate the combined tables/manifest.')
        print('Skipping Mamba runtime installation for this final aggregation pass because no Full-model inference is required.')
    else:
        print('Missing robustness stresses will be generated and published one at a time:', missing_robustness_stresses)
        if missing_robustness_references:
            print('MiniRocket reference artifacts will be regenerated:', [path.as_posix() for path in missing_robustness_references])
    print('Robustness metric cache dir:', ROBUSTNESS_METRIC_CACHE_DIR)

robustness_command = (
    'python -u scripts/revision/12_robustness_stress.py '
    f'--stress-tests {ROBUSTNESS_STRESS_TESTS} '
    f'--limit-records {ROBUSTNESS_LIMIT_RECORDS} '
    f'--checkpoint-kind final_ema --expected-checkpoint-kind final_ema '
    '--oof-run-manifest reports/revision/manifests/oof_final_ema_prediction_run_manifest.json '
    f'--threshold 0.5 --n-bins 15 --n-boot {ROBUSTNESS_N_BOOT} '
    f'--metric-cache-dir "{ROBUSTNESS_METRIC_CACHE_DIR}" '
    f'--batch-size {ROBUSTNESS_BATCH_SIZE} --num-workers {ROBUSTNESS_NUM_WORKERS} '
    f'--minirocket-feature-device {ROBUSTNESS_MINIROCKET_FEATURE_DEVICE} '
    '--minirocket-device auto --minirocket-epochs 20 '
    '--reuse-existing --require-existing-stress-predictions --reuse-metric-cache '
    '--bootstrap-progress-every 100 --reuse-minirocket-heads --save-perturbed-caches'
)
if ROBUSTNESS_ALLOW_TF32:
    robustness_command += ' --allow-tf32'

if needs_robustness_regeneration and not robustness_predictions_complete:
    import importlib
    import json
    missing_runtime_modules = []
    for module_name in ['mamba_ssm', 'causal_conv1d']:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            missing_runtime_modules.append((module_name, repr(exc)))
    if missing_runtime_modules:
        print('Mamba runtime modules are missing; running canonical installer from Notebook 02:', missing_runtime_modules)
        installer_nb_path = REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb'
        installer_nb = json.loads(installer_nb_path.read_text(encoding='utf-8'))
        installer_candidates = []
        for notebook_cell in installer_nb['cells']:
            if notebook_cell.get('cell_type') != 'code':
                continue
            source = ''.join(notebook_cell.get('source', []))
            if "MAMBA_INSTALLER_CAPABILITY = 'ecg_ramba_mamba_installer_v1'" in source and 'MAMBA_INSTALLER_SCHEMA_VERSION = 1' in source:
                installer_candidates.append(source)
        installer_source = installer_candidates[0] if len(installer_candidates) == 1 else None
        if installer_source is None:
            raise RuntimeError(f'Could not uniquely locate canonical Mamba installer cell in Notebook 02; candidate_count={len(installer_candidates)}.')
        exec(compile(installer_source, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
        importlib.invalidate_caches()
    for module_name in ['mamba_ssm', 'causal_conv1d']:
        loaded_module = importlib.import_module(module_name)
        print('OK import:', module_name, getattr(loaded_module, '__file__', 'built-in'))

if needs_robustness_regeneration:
    if not Path('scripts/revision/12_robustness_stress.py').exists():
        raise FileNotFoundError('Missing scripts/revision/12_robustness_stress.py')

    if not robustness_predictions_complete:
        stresses_to_generate = list(missing_robustness_stresses)
        if not stresses_to_generate and missing_robustness_references:
            stresses_to_generate = [robustness_stress_list[0]]
        prediction_cache_dir = (
            DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' /
            'robustness_prediction_generation_metric_cache'
        )
        for stress_name in stresses_to_generate:
            prediction_command = robustness_command.replace(
                f'--stress-tests {ROBUSTNESS_STRESS_TESTS}',
                f'--stress-tests {stress_name}',
            ).replace(
                f'--n-boot {ROBUSTNESS_N_BOOT}',
                '--n-boot 1',
            ).replace(
                f'--metric-cache-dir "{ROBUSTNESS_METRIC_CACHE_DIR}"',
                f'--metric-cache-dir "{prediction_cache_dir}"',
            ).replace(' --require-existing-stress-predictions', '')
            if FORCE_RERUN_ROBUSTNESS_STRESS:
                prediction_command = prediction_command.replace('--reuse-existing', '--no-reuse-existing')
            print(f'Generating canonical Full/MiniRocket stress predictions for {stress_name}.')
            run(
                prediction_command,
                log_path=f'reports/revision/logs/robustness_prediction_{stress_name}.log',
            )
            if 'MIRROR_REVISION_ROOT' in globals():
                run(
                    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --mirror-root "{MIRROR_REVISION_ROOT}"',
                    log_path=f'reports/revision/logs/robustness_prediction_{stress_name}_mirror_publish.log',
                )

    # Final low-memory n_boot=1000 aggregation reuses all generated predictions.
    run(robustness_command, log_path='reports/revision/logs/robustness_stress.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/robustness_final_aggregate_mirror_publish.log',
        )
elif robustness_outputs_ready:
    print('Reusing existing robustness stress artifacts. Set FORCE_RERUN_ROBUSTNESS_STRESS=True to recompute.')
else:
    robustness_rows = []
    for analysis_name, perturbation in [
        ('SNR 20 dB noise', 'snr20db'),
        ('SNR 10 dB noise', 'snr10db'),
        ('SNR 5 dB noise', 'snr5db'),
        ('Random 3-lead dropout', 'random_3_lead_dropout'),
        ('V1-V6 dropout', 'precordial_dropout'),
        ('500Hz to 250Hz sampling shift', 'resample_250hz'),
    ]:
        robustness_rows.append({
            'analysis_name': analysis_name,
            'perturbation': perturbation,
            'dataset': 'chapman_oof',
            'status': 'blocked_not_run',
            'clean_metric_reference': str(calibration_path),
            'perturbed_metric_value': math.nan,
            'absolute_delta': math.nan,
            'relative_delta': math.nan,
            'evidence_path': '',
            'blocker': 'Robustness runner is implemented but RUN_ROBUSTNESS_STRESS=False and no prior artifacts were restored.',
            'safe_wording': 'Do not claim robustness under this perturbation until the runner is executed and artifacts are frozen.',
        })
    robustness_df = pd.DataFrame(robustness_rows)
    robustness_outputs['summary_csv'].parent.mkdir(parents=True, exist_ok=True)
    robustness_outputs['table_csv'].parent.mkdir(parents=True, exist_ok=True)
    robustness_df.to_csv(robustness_outputs['summary_csv'], index=False)
    robustness_df.to_csv(robustness_outputs['table_csv'], index=False)
    print('Wrote blocked robustness ledger:', robustness_outputs['summary_csv'])

for name, path in robustness_outputs.items():
    print(f'{name:18s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

robustness_df = pd.read_csv(robustness_outputs['summary_csv'])
if ROBUSTNESS_FINAL_AGGREGATION_PASS:
    actual_stresses = set(robustness_df['stress_test'].astype(str))
    expected_stresses = set(expected_full_robustness_stresses)
    actual_metrics = set(robustness_df['metric'].astype(str)) if 'metric' in robustness_df.columns else set()
    expected_rows = len(expected_full_robustness_stresses) * len(expected_robustness_metrics)
    if actual_stresses != expected_stresses:
        raise RuntimeError(f'Final robustness summary has wrong stress set: {sorted(actual_stresses)}')
    if actual_metrics != expected_robustness_metrics:
        raise RuntimeError(f'Final robustness summary has wrong metric set: {sorted(actual_metrics)}')
    if len(robustness_df) != expected_rows:
        raise RuntimeError(f'Final robustness summary should have {expected_rows} rows, found {len(robustness_df)}')
    print(f'Final robustness aggregation validated: {len(robustness_df)} rows across {len(expected_full_robustness_stresses)} stresses and {len(expected_robustness_metrics)} metrics.')

display(robustness_df.sort_values(['stress_test', 'metric']).reset_index(drop=True))


## Comparator Stress Prediction Generation

Default direct-run behavior follows `ECG_RAMBA_ROBUSTNESS_MULTI_PROFILE=canonical_resume`, which requires all six registered stresses for ResNet1D/CNN, Raw Mamba, and Transformer ECG. Existing prediction artifacts are reused only after exact stress-specification, raw-cache, fold, checkpoint-SHA, OOF, and freeze-contract validation. Use A100 High-RAM only when a required prediction is missing or stale; if all requested outputs authenticate against the canonical Drive mirror, this cell skips GPU inference.
Generate stressed prediction artifacts for ResNet1D/CNN, Raw Mamba, and Transformer ECG from checkpoints saved by Notebook 04. This cell is inference-only and will not retrain. It restores checkpoints and existing stress artifacts from the Drive mirror, then skips generation when all requested outputs are already complete.


In [ ]:
RUN_COMPARATOR_STRESS_PREDICTIONS = True
FORCE_RERUN_COMPARATOR_STRESS_PREDICTIONS = False
COMPARATOR_STRESS_COMPARATORS = os.environ.get('ECG_RAMBA_COMPARATOR_STRESS_COMPARATORS', 'resnet,raw_mamba,transformer')

# Keep this aligned with the multi-comparator profile so direct reruns do not generate unnecessary stresses.
_COMPARATOR_STRESS_PROFILE = os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_PROFILE', 'canonical_resume')
_COMPARATOR_STRESS_REVIEWER_STRESSES = os.environ.get(
    'ECG_RAMBA_ROBUSTNESS_MULTI_REVIEWER_STRESSES',
    'snr5db,precordial_dropout',
)
_COMPARATOR_STRESS_SINGLE_STRESS = os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_SINGLE_STRESS', 'snr20db')
if _COMPARATOR_STRESS_PROFILE in {'reviewer_minimal', 'core_final'}:
    _COMPARATOR_STRESS_DEFAULT_STRESSES = _COMPARATOR_STRESS_REVIEWER_STRESSES
elif _COMPARATOR_STRESS_PROFILE == 'single_stress_final':
    _COMPARATOR_STRESS_DEFAULT_STRESSES = _COMPARATOR_STRESS_SINGLE_STRESS
else:
    _COMPARATOR_STRESS_DEFAULT_STRESSES = ROBUSTNESS_STRESS_TESTS
COMPARATOR_STRESS_TESTS = os.environ.get('ECG_RAMBA_COMPARATOR_STRESS_TESTS', _COMPARATOR_STRESS_DEFAULT_STRESSES)

# A100 High-RAM handles 512 comfortably in current checkpoints; set ECG_RAMBA_COMPARATOR_STRESS_BATCH_SIZE=256 on smaller GPUs.
COMPARATOR_STRESS_BATCH_SIZE = int(os.environ.get('ECG_RAMBA_COMPARATOR_STRESS_BATCH_SIZE', '512'))
COMPARATOR_STRESS_NUM_WORKERS = 0
COMPARATOR_STRESS_STRICT = _COMPARATOR_STRESS_PROFILE in {'full_final', 'canonical_resume'}

SUPPORTED_COMPARATOR_STRESS = {'resnet', 'raw_mamba', 'transformer'}
COMPARATOR_STRESS_STEMS = {
    'resnet': 'resnet1d_cnn',
    'raw_mamba': 'raw_mamba',
    'transformer': 'transformer_ecg',
}
COMPARATOR_STRESS_CHECKPOINT_DIRS = {
    'resnet': CANONICAL_CHECKPOINT_ROOT / 'resnet1d_cnn_checkpoints',
    'raw_mamba': CANONICAL_CHECKPOINT_ROOT / 'raw_mamba_checkpoints',
    'transformer': CANONICAL_CHECKPOINT_ROOT / 'transformer_ecg_checkpoints',
}
COMPARATOR_STRESS_CHECKPOINTS = {
    'resnet': [COMPARATOR_STRESS_CHECKPOINT_DIRS['resnet'] / f'fold{i}_resnet1d_cnn_final.pt' for i in range(1, 6)],
    'raw_mamba': [COMPARATOR_STRESS_CHECKPOINT_DIRS['raw_mamba'] / f'fold{i}_raw_mamba_final_ema.pt' for i in range(1, 6)],
    'transformer': [COMPARATOR_STRESS_CHECKPOINT_DIRS['transformer'] / f'fold{i}_transformer_ecg_final.pt' for i in range(1, 6)],
}

comparator_stress_parts = [item.strip() for item in COMPARATOR_STRESS_COMPARATORS.split(',') if item.strip()]
unsupported_comparator_stress = sorted(set(comparator_stress_parts) - SUPPORTED_COMPARATOR_STRESS)
if unsupported_comparator_stress:
    raise ValueError(
        'Unsupported comparator stress request: '
        + ', '.join(unsupported_comparator_stress)
        + '. Current generator supports only resnet,raw_mamba,transformer.'
    )

comparator_stress_list = [item.strip() for item in COMPARATOR_STRESS_TESTS.split(',') if item.strip()]
comparator_checkpoint_candidates = [
    path
    for comparator in comparator_stress_parts
    for path in COMPARATOR_STRESS_CHECKPOINTS[comparator]
]
comparator_stress_outputs = [
    revision_root / 'predictions' / f'robustness_{COMPARATOR_STRESS_STEMS[comparator]}_{stress}_predictions.npz'
    for stress in comparator_stress_list
    for comparator in comparator_stress_parts
]
comparator_stress_manifest = revision_root / 'manifests' / 'comparator_stress_prediction_manifest.json'

print('Comparator stress profile:', _COMPARATOR_STRESS_PROFILE)
print('Comparator stress comparators:', comparator_stress_parts)
print('Comparator stress tests:', comparator_stress_list)
print('Comparator stress batch size:', COMPARATOR_STRESS_BATCH_SIZE)
print('Comparator checkpoint source: canonical Drive mirror', CANONICAL_CHECKPOINT_ROOT)

# Checkpoints stay in the canonical mirror and are read directly; only smaller prediction artifacts are restored.
if '_restore_revision_artifacts_from_drive' in globals():
    _restore_revision_artifacts_from_drive(
        comparator_stress_outputs + [comparator_stress_manifest],
        'Comparator stress prediction artifacts',
    )

print('Comparator checkpoint status:')
checkpoint_rows = []
for path in comparator_checkpoint_candidates:
    row = {
        'path': path.as_posix(),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
    }
    checkpoint_rows.append(row)
    print(f"  {row['path']}: exists={row['exists']} size={row['size_bytes']}")
checkpoint_status_df = pd.DataFrame(checkpoint_rows)
missing_checkpoint_paths = checkpoint_status_df.loc[~checkpoint_status_df['exists'], 'path'].astype(str).tolist()

print('Comparator stress output status:')
output_rows = []
for path in comparator_stress_outputs:
    row = {
        'path': path.as_posix(),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
    }
    output_rows.append(row)
    print(f"  {row['path']}: exists={row['exists']} size={row['size_bytes']}")
comparator_stress_status_df = pd.DataFrame(output_rows)
missing_comparator_stress_outputs = comparator_stress_status_df.loc[~comparator_stress_status_df['exists'], 'path'].astype(str).tolist()

comparator_stress_command = (
    'python -u scripts/revision/23_generate_comparator_stress_predictions.py '
    f'--comparators {COMPARATOR_STRESS_COMPARATORS} '
    f'--stress-tests {COMPARATOR_STRESS_TESTS} '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    f'--resnet-checkpoint-dir "{COMPARATOR_STRESS_CHECKPOINT_DIRS["resnet"]}" '
    f'--raw-mamba-checkpoint-dir "{COMPARATOR_STRESS_CHECKPOINT_DIRS["raw_mamba"]}" '
    f'--transformer-checkpoint-dir "{COMPARATOR_STRESS_CHECKPOINT_DIRS["transformer"]}" '
    f'--batch-size {COMPARATOR_STRESS_BATCH_SIZE} --num-workers {COMPARATOR_STRESS_NUM_WORKERS} '
    '--device cuda --amp --allow-tf32 --reuse-existing'
)
if FORCE_RERUN_COMPARATOR_STRESS_PREDICTIONS:
    comparator_stress_command += ' --no-reuse-existing'
if COMPARATOR_STRESS_STRICT:
    comparator_stress_command += ' --strict'

comparator_stress_ran = False
print('Comparator stress command:', comparator_stress_command)

# File existence is insufficient for reuse. Validate OOF/freeze, record/class order,
# aggregation, and checkpoint SHA contracts before deciding whether GPU inference is needed.
stresses_to_run = list(comparator_stress_list) if FORCE_RERUN_COMPARATOR_STRESS_PREDICTIONS else []
stale_pairs = []
comparator_stress_preflight = {}
if not missing_checkpoint_paths and Path('scripts/revision/23_generate_comparator_stress_predictions.py').exists():
    preflight_command = comparator_stress_command.replace(' --strict', '').replace(' --no-reuse-existing', '') + ' --finalize-manifest-only'
    run(preflight_command, log_path='reports/revision/logs/comparator_stress_manifest_preflight.log')
    comparator_stress_preflight = json.loads(comparator_stress_manifest.read_text(encoding='utf-8'))
    preflight_missing = list(comparator_stress_preflight.get('missing') or [])
    authenticated_pairs = {
        (str(row.get('comparator')), str(row.get('stress')))
        for row in (comparator_stress_preflight.get('artifacts') or [])
    }
    for issue in preflight_missing:
        pair_token = str(issue).split(' ', 1)[0]
        if '/' in pair_token:
            comparator_name, stress_name = pair_token.split('/', 1)
            stale_pairs.append((comparator_name, stress_name))
            if stress_name in comparator_stress_list and stress_name not in stresses_to_run:
                stresses_to_run.append(stress_name)
    print('Comparator stress provenance preflight:', comparator_stress_preflight.get('status'))
    print('Authenticated comparator/stress artifacts:', len(authenticated_pairs), '/', len(comparator_stress_outputs))
    if stale_pairs:
        print('Missing/stale comparator stress pairs:', stale_pairs)
        print('Only affected stress groups will run; valid comparator outputs inside each group will be reused.')
    elif not FORCE_RERUN_COMPARATOR_STRESS_PREDICTIONS:
        print('All requested comparator stress artifacts passed provenance validation; GPU inference will be skipped.')

needs_comparator_stress_inference = RUN_COMPARATOR_STRESS_PREDICTIONS and bool(stresses_to_run)

if needs_comparator_stress_inference and not missing_checkpoint_paths:
    import importlib.util
    import torch

    print('Comparator stress inference requires CUDA because the command uses --device cuda.')
    print('torch.cuda.is_available():', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    else:
        raise RuntimeError('Comparator stress predictions are missing and CUDA is unavailable. Switch to A100/T4 GPU or restore completed prediction artifacts from the mirror.')

    missing_runtime = [
        module for module in ['mamba_ssm', 'causal_conv1d']
        if importlib.util.find_spec(module) is None
    ]
    raw_mamba_needs_inference = FORCE_RERUN_COMPARATOR_STRESS_PREDICTIONS or any(
        comparator_name == 'raw_mamba' for comparator_name, _ in stale_pairs
    )
    print('Raw Mamba stress inference required:', raw_mamba_needs_inference)
    if raw_mamba_needs_inference and missing_runtime:
        import importlib
        import json

        print('Raw Mamba stress inference needs Mamba runtime; installing from canonical Notebook 02 installer inside Notebook 05. Missing:', missing_runtime)
        installer_nb_path = REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb'
        installer_nb = json.loads(installer_nb_path.read_text(encoding='utf-8'))
        installer_candidates = []
        for notebook_cell in installer_nb['cells']:
            if notebook_cell.get('cell_type') != 'code':
                continue
            source = ''.join(notebook_cell.get('source', []))
            if "MAMBA_INSTALLER_CAPABILITY = 'ecg_ramba_mamba_installer_v1'" in source and 'MAMBA_INSTALLER_SCHEMA_VERSION = 1' in source:
                installer_candidates.append(source)
        installer_source = installer_candidates[0] if len(installer_candidates) == 1 else None
        if installer_source is None:
            raise RuntimeError(f'Could not uniquely locate canonical Mamba installer cell in Notebook 02; candidate_count={len(installer_candidates)}. Pull latest main and rerun this Notebook 05 Setup cell.')
        exec(compile(installer_source, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
        importlib.invalidate_caches()
        still_missing = [
            module for module in ['mamba_ssm', 'causal_conv1d']
            if importlib.util.find_spec(module) is None
        ]
        if still_missing:
            raise ImportError(
                'Mamba installer completed but required modules are still missing: '
                + ', '.join(still_missing)
                + '. If the installer requested a runtime restart, restart once and rerun Notebook 05 from Setup.'
            )
        print('Mamba runtime is available after Notebook 05 direct installer.')

if not RUN_COMPARATOR_STRESS_PREDICTIONS:
    print('Comparator stress prediction generation disabled. Existing artifacts, if present, remain usable.')
elif missing_checkpoint_paths:
    raise FileNotFoundError(
        'Comparator stress inference needs saved checkpoints before generating missing outputs. Missing checkpoints: '
        + '; '.join(missing_checkpoint_paths)
        + '. Complete/publish the missing Notebook 04 fold checkpoint(s) in the canonical mirror, then rerun this cell.'
    )
else:
    if not Path('scripts/revision/23_generate_comparator_stress_predictions.py').exists():
        raise FileNotFoundError('Missing scripts/revision/23_generate_comparator_stress_predictions.py')

    # Run and publish one stale stress group at a time. The runner reuses
    # provenance-valid comparators and recomputes only stale members.
    if not stresses_to_run:
        print('All requested comparator stress prediction artifacts are provenance-valid; skipping GPU inference.')
    for stress in stresses_to_run:
        stress_command = comparator_stress_command.replace(
            f'--stress-tests {COMPARATOR_STRESS_TESTS}',
            f'--stress-tests {stress}',
        )
        print(f'Running comparator stress {stress} with immediate canonical mirror publish.')
        run(stress_command, log_path=f'reports/revision/logs/comparator_stress_{stress}.log')
        comparator_stress_ran = True
        if 'MIRROR_REVISION_ROOT' in globals():
            run(
                f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --refresh-existing-prefix predictions/robustness_ --mirror-root "{MIRROR_REVISION_ROOT}"',
                log_path=f'reports/revision/logs/comparator_stress_{stress}_mirror_publish.log',
            )

    # Rebuild one combined manifest without loading/perturbing raw ECG again.
    finalize_command = comparator_stress_command + ' --finalize-manifest-only'
    run(finalize_command, log_path='reports/revision/logs/comparator_stress_manifest_finalize.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/comparator_stress_manifest_mirror_publish.log',
        )

print('Comparator stress final status:')
for path in comparator_stress_outputs:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
print(f'{comparator_stress_manifest}: exists={comparator_stress_manifest.exists()} size={comparator_stress_manifest.stat().st_size if comparator_stress_manifest.exists() else None}')

missing_after_comparator_stress = [
    path.as_posix() for path in comparator_stress_outputs if not (path.exists() and path.stat().st_size > 0)
]
if missing_after_comparator_stress:
    print('WARNING: missing comparator stress outputs remain:', '; '.join(missing_after_comparator_stress))
else:
    print('All requested comparator stress outputs are ready for multi-comparator aggregation.')


## Multi-Comparator Robustness Ledger

Aggregate existing stress predictions across Full ECG-RAMBA, MiniRocket-only, ResNet1D/CNN, Raw Mamba, and Transformer ECG. This cell does not run model inference; CPU is enough once the stress prediction `.npz` files exist.

Default direct-run profile is `canonical_resume`: all six stresses, all five registered metrics (`pr_auc_macro`, `roc_auc_macro`, `f1_macro`, `brier_macro`, `ece_macro`), and `n_boot=1000`. It runs one stress at a time, publishes each stage, and then assembles the canonical ledger. Numeric metric-cache reuse is allowed only when the four paired prediction SHA values and analysis settings match.

This ledger uses paired record bootstrap CIs that are nominal, pointwise, and unadjusted for the full comparison family. They condition on fixed folds, trained checkpoints, and one fixed perturbation seed; therefore manuscript wording must remain stress/metric/comparator-specific and must not claim general or family-wise robustness superiority. CPU High-RAM is sufficient for this cell; `canonical_resume` is safer than `full_final` for interrupted Colab sessions.


In [ ]:
RUN_ROBUSTNESS_MULTICOMPARATOR = True
FORCE_RERUN_ROBUSTNESS_MULTICOMPARATOR = False

# Runtime profiles:
# - reviewer_minimal: fast screening only; 2 representative stresses, 3 metrics, n_boot=200.
# - full_screening: all 6 stresses, 3 metrics, n_boot=200; broader but slower.
# - single_stress_final: one stress, configurable metrics, n_boot=1000; use to finalize one row family at a time and reuse canonical final caches.
# - core_final: configured representative stresses, all 5 metrics by default, n_boot=1000; can still take many hours.
# - full_final: all stresses, all metrics, n_boot=1000 in one command.
# - canonical_resume: reviewer-complete default; runs one stress at a time with durable per-metric cache, then assembles the canonical ledger.
ROBUSTNESS_MULTI_RUN_PROFILE = os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_PROFILE', 'canonical_resume')
ROBUSTNESS_MULTI_SINGLE_STRESS = os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_SINGLE_STRESS', 'snr20db')
ROBUSTNESS_MULTI_REVIEWER_STRESSES = os.environ.get(
    'ECG_RAMBA_ROBUSTNESS_MULTI_REVIEWER_STRESSES',
    'snr5db,precordial_dropout',
)
ROBUSTNESS_MULTI_COMPARATORS_DEFAULT = os.environ.get(
    'ECG_RAMBA_ROBUSTNESS_MULTI_COMPARATORS',
    'full,minirocket,resnet,raw_mamba,transformer',
)
ROBUSTNESS_MULTI_SCREENING_N_BOOT = int(os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_SCREENING_N_BOOT', '200'))
ROBUSTNESS_MULTI_FINAL_N_BOOT = int(os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_FINAL_N_BOOT', '1000'))
ROBUSTNESS_MULTI_SCREENING_METRICS = os.environ.get(
    'ECG_RAMBA_ROBUSTNESS_MULTI_SCREENING_METRICS',
    'pr_auc_macro,roc_auc_macro,f1_macro',
)
ROBUSTNESS_MULTI_FINAL_METRICS = os.environ.get(
    'ECG_RAMBA_ROBUSTNESS_MULTI_FINAL_METRICS',
    'pr_auc_macro,roc_auc_macro,f1_macro,brier_macro,ece_macro',
)
ROBUSTNESS_MULTI_CPU_COUNT = max(1, os.cpu_count() or 1)
ROBUSTNESS_MULTI_DEFAULT_BOOTSTRAP_JOBS = min(8, ROBUSTNESS_MULTI_CPU_COUNT)
ROBUSTNESS_MULTI_BOOTSTRAP_JOBS = int(os.environ.get(
    'ECG_RAMBA_ROBUSTNESS_MULTI_BOOTSTRAP_JOBS',
    str(ROBUSTNESS_MULTI_DEFAULT_BOOTSTRAP_JOBS),
))
ROBUSTNESS_MULTI_INNER_THREADS = int(os.environ.get('ECG_RAMBA_ROBUSTNESS_MULTI_INNER_THREADS', '1'))
if ROBUSTNESS_MULTI_BOOTSTRAP_JOBS < 1 or ROBUSTNESS_MULTI_INNER_THREADS < 1:
    raise ValueError('Bootstrap jobs and inner-thread count must both be at least 1.')
# The bootstrap subprocess uses Python workers around NumPy reductions. Keep
# BLAS/OpenMP single-threaded to avoid worker x BLAS oversubscription.
for _thread_env in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_thread_env] = str(ROBUSTNESS_MULTI_INNER_THREADS)
ROBUSTNESS_MULTI_STRICT = ROBUSTNESS_MULTI_RUN_PROFILE in {'full_final', 'canonical_resume'}
ROBUSTNESS_MULTI_WRITE_CANONICAL = False
# Keep final-profile cache on the canonical cache path so expensive interrupted n_boot=1000 rows are reused.
ROBUSTNESS_MULTI_REUSE_CANONICAL_FINAL_CACHE = True

VALID_ROBUSTNESS_MULTI_PROFILES = {
    'reviewer_minimal',
    'full_screening',
    'single_stress_final',
    'core_final',
    'full_final',
    'canonical_resume',
}
if ROBUSTNESS_MULTI_RUN_PROFILE not in VALID_ROBUSTNESS_MULTI_PROFILES:
    raise ValueError(
        f'Unknown ROBUSTNESS_MULTI_RUN_PROFILE={ROBUSTNESS_MULTI_RUN_PROFILE!r}. '
        f'Choose one of {sorted(VALID_ROBUSTNESS_MULTI_PROFILES)}.'
    )

if ROBUSTNESS_MULTI_RUN_PROFILE == 'reviewer_minimal':
    ROBUSTNESS_MULTI_COMPARATORS = ROBUSTNESS_MULTI_COMPARATORS_DEFAULT
    ROBUSTNESS_MULTI_STRESSES = ROBUSTNESS_MULTI_REVIEWER_STRESSES
    ROBUSTNESS_MULTI_N_BOOT = ROBUSTNESS_MULTI_SCREENING_N_BOOT
    ROBUSTNESS_MULTI_METRICS = ROBUSTNESS_MULTI_SCREENING_METRICS
    ROBUSTNESS_MULTI_OUTPUT_STEM = 'robustness_multicomparator_reviewer_minimal'
elif ROBUSTNESS_MULTI_RUN_PROFILE == 'full_screening':
    ROBUSTNESS_MULTI_COMPARATORS = ROBUSTNESS_MULTI_COMPARATORS_DEFAULT
    ROBUSTNESS_MULTI_STRESSES = ROBUSTNESS_STRESS_TESTS
    ROBUSTNESS_MULTI_N_BOOT = ROBUSTNESS_MULTI_SCREENING_N_BOOT
    ROBUSTNESS_MULTI_METRICS = ROBUSTNESS_MULTI_SCREENING_METRICS
    ROBUSTNESS_MULTI_OUTPUT_STEM = 'robustness_multicomparator_full_screening'
elif ROBUSTNESS_MULTI_RUN_PROFILE == 'single_stress_final':
    ROBUSTNESS_MULTI_COMPARATORS = ROBUSTNESS_MULTI_COMPARATORS_DEFAULT
    ROBUSTNESS_MULTI_STRESSES = ROBUSTNESS_MULTI_SINGLE_STRESS
    ROBUSTNESS_MULTI_N_BOOT = ROBUSTNESS_MULTI_FINAL_N_BOOT
    ROBUSTNESS_MULTI_METRICS = ROBUSTNESS_MULTI_FINAL_METRICS
    safe_stress = ''.join(ch if ch.isalnum() or ch in {'_', '-'} else '_' for ch in ROBUSTNESS_MULTI_SINGLE_STRESS)
    ROBUSTNESS_MULTI_OUTPUT_STEM = f'robustness_multicomparator_{safe_stress}_final'
elif ROBUSTNESS_MULTI_RUN_PROFILE == 'core_final':
    ROBUSTNESS_MULTI_COMPARATORS = ROBUSTNESS_MULTI_COMPARATORS_DEFAULT
    ROBUSTNESS_MULTI_STRESSES = ROBUSTNESS_MULTI_REVIEWER_STRESSES
    ROBUSTNESS_MULTI_N_BOOT = ROBUSTNESS_MULTI_FINAL_N_BOOT
    ROBUSTNESS_MULTI_METRICS = ROBUSTNESS_MULTI_FINAL_METRICS
    ROBUSTNESS_MULTI_OUTPUT_STEM = 'robustness_multicomparator_core_final'
else:  # full_final or canonical_resume
    ROBUSTNESS_MULTI_COMPARATORS = ROBUSTNESS_MULTI_COMPARATORS_DEFAULT
    ROBUSTNESS_MULTI_STRESSES = ROBUSTNESS_STRESS_TESTS
    ROBUSTNESS_MULTI_N_BOOT = ROBUSTNESS_MULTI_FINAL_N_BOOT
    ROBUSTNESS_MULTI_METRICS = 'pr_auc_macro,roc_auc_macro,f1_macro,brier_macro,ece_macro'
    ROBUSTNESS_MULTI_OUTPUT_STEM = 'robustness_multicomparator'
    ROBUSTNESS_MULTI_WRITE_CANONICAL = True

if (
    ROBUSTNESS_MULTI_RUN_PROFILE in {'single_stress_final', 'core_final', 'full_final', 'canonical_resume'}
    and ROBUSTNESS_MULTI_REUSE_CANONICAL_FINAL_CACHE
):
    ROBUSTNESS_MULTI_METRIC_CACHE_DIR = (
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' /
        'robustness_multicomparator_metric_cache'
    )
else:
    ROBUSTNESS_MULTI_METRIC_CACHE_DIR = (
        DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' /
        f'{ROBUSTNESS_MULTI_OUTPUT_STEM}_metric_cache'
    )

ROBUSTNESS_MULTI_STEMS = {
    'full': 'full',
    'minirocket': 'minirocket',
    'resnet': 'resnet1d_cnn',
    'raw_mamba': 'raw_mamba',
    'transformer': 'transformer_ecg',
}
robustness_multi_parts = [item.strip() for item in ROBUSTNESS_MULTI_COMPARATORS.split(',') if item.strip()]
robustness_multi_stress_list = [item.strip() for item in ROBUSTNESS_MULTI_STRESSES.split(',') if item.strip()]
robustness_multi_metric_list = [item.strip() for item in ROBUSTNESS_MULTI_METRICS.split(',') if item.strip()]
unknown_multi_parts = sorted(set(robustness_multi_parts) - set(ROBUSTNESS_MULTI_STEMS))
if unknown_multi_parts:
    raise ValueError(f'Unsupported robustness multi-comparator(s): {unknown_multi_parts}')
if 'full' not in robustness_multi_parts:
    raise ValueError('ROBUSTNESS_MULTI_COMPARATORS must include full because all paired degradation claims use Full ECG-RAMBA as reference.')
if not robustness_multi_stress_list:
    raise ValueError('ROBUSTNESS_MULTI_STRESSES is empty.')
if not robustness_multi_metric_list:
    raise ValueError('ROBUSTNESS_MULTI_METRICS is empty.')

if ROBUSTNESS_MULTI_OUTPUT_STEM == 'robustness_multicomparator' or ROBUSTNESS_MULTI_WRITE_CANONICAL:
    multi_robustness_outputs = {
        'summary': revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
        'pairwise': revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
        'table': revision_root / 'tables' / 'table_robustness_multicomparator.csv',
        'manifest': revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
    }
else:
    multi_robustness_outputs = {
        'summary': revision_root / 'metrics' / f'{ROBUSTNESS_MULTI_OUTPUT_STEM}_summary.csv',
        'pairwise': revision_root / 'metrics' / f'{ROBUSTNESS_MULTI_OUTPUT_STEM}_pairwise.json',
        'table': revision_root / 'tables' / f'table_{ROBUSTNESS_MULTI_OUTPUT_STEM}.csv',
        'manifest': revision_root / 'manifests' / f'{ROBUSTNESS_MULTI_OUTPUT_STEM}_manifest.json',
    }

# Comparator sidecars are profile-scoped. Canonical names are reserved for the full-final ledger.
_multi_sidecar_profile = ROBUSTNESS_MULTI_OUTPUT_STEM.replace('robustness_multicomparator', '', 1).strip('_')
_multi_sidecar_suffix = f'_{_multi_sidecar_profile}' if _multi_sidecar_profile else ''
for _comparator in ['resnet', 'raw_mamba', 'transformer']:
    if _comparator in robustness_multi_parts:
        multi_robustness_outputs[f'sidecar_{_comparator}'] = (
            revision_root / 'metrics' / f'robustness_full_vs_{_comparator}{_multi_sidecar_suffix}_comparison.json'
        )

multi_robustness_command = (
    'python -u scripts/revision/21_robustness_multicomparator.py '
    f'--comparators {ROBUSTNESS_MULTI_COMPARATORS} '
    f'--stress-tests {ROBUSTNESS_MULTI_STRESSES} '
    f'--metrics {ROBUSTNESS_MULTI_METRICS} '
    f'--threshold 0.5 --n-bins 15 --n-boot {ROBUSTNESS_MULTI_N_BOOT} '
    f'--bootstrap-jobs {ROBUSTNESS_MULTI_BOOTSTRAP_JOBS} '
    f'--metric-cache-dir "{ROBUSTNESS_MULTI_METRIC_CACHE_DIR}" --reuse-metric-cache '
    f'--out-summary {multi_robustness_outputs["summary"]} '
    f'--out-pairwise {multi_robustness_outputs["pairwise"]} '
    f'--out-table {multi_robustness_outputs["table"]} '
    f'--out-manifest {multi_robustness_outputs["manifest"]}'
)
if ROBUSTNESS_MULTI_STRICT:
    multi_robustness_command += ' --strict'

print('Multi-comparator robustness profile:', ROBUSTNESS_MULTI_RUN_PROFILE)
print('  comparators:', robustness_multi_parts)
print('  stresses   :', robustness_multi_stress_list)
print('  metrics    :', robustness_multi_metric_list)
print('  n_boot     :', ROBUSTNESS_MULTI_N_BOOT)
print('  bootstrap_jobs:', ROBUSTNESS_MULTI_BOOTSTRAP_JOBS)
print('  detected_cpu_count:', ROBUSTNESS_MULTI_CPU_COUNT)
print('  inner_threads:', ROBUSTNESS_MULTI_INNER_THREADS)
print('  bootstrap_engine: paired record resampling with cached score ranks and weighted counts')
print('  total paired metric jobs:', (len(robustness_multi_parts) - 1) * len(robustness_multi_stress_list) * len(robustness_multi_metric_list))
print('  hardware   : CPU-bound aggregation; a GPU does not accelerate this cell')
print('  outputs    :', {k: v.as_posix() for k, v in multi_robustness_outputs.items()})
if ROBUSTNESS_MULTI_N_BOOT < 1000:
    print('NOTE: this is a reviewer-minimal/screening bootstrap pass. Use single_stress_final/core_final/full_final for n_boot=1000 claim-ready rows.')
if ROBUSTNESS_MULTI_RUN_PROFILE in {'single_stress_final', 'core_final', 'full_final', 'canonical_resume'} and ROBUSTNESS_MULTI_REUSE_CANONICAL_FINAL_CACHE:
    print('Final-profile cache reuse: using canonical robustness_multicomparator_metric_cache for interrupted n_boot=1000 rows.')
if ROBUSTNESS_MULTI_RUN_PROFILE == 'full_final':
    print('WARNING: full_final runs the complete canonical grid in one command.')
if ROBUSTNESS_MULTI_RUN_PROFILE == 'canonical_resume':
    print('Canonical resume mode: each stress is cached and published separately before the final canonical assembly.')

robustness_multi_clean_predictions = [
    revision_root / 'predictions' / {
        'full': 'oof_final_ema_predictions.npz',
        'minirocket': 'robustness_minirocket_clean_ref_predictions.npz',
        'resnet': 'resnet1d_cnn_oof_predictions.npz',
        'raw_mamba': 'raw_mamba_oof_predictions.npz',
        'transformer': 'transformer_ecg_oof_predictions.npz',
    }[comparator]
    for comparator in robustness_multi_parts
]
robustness_multi_contract_artifacts = [
    revision_root / 'manifests' / 'oof_final_ema_prediction_run_manifest.json',
    revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json',
    revision_root / 'manifests' / 'robustness_minirocket_heads_manifest.json',
]
_learned_manifest_names = {
    'resnet': 'resnet1d_cnn_baseline_manifest.json',
    'raw_mamba': 'raw_mamba_baseline_manifest.json',
    'transformer': 'transformer_ecg_baseline_manifest.json',
}
robustness_multi_contract_artifacts.extend(
    revision_root / 'manifests' / _learned_manifest_names[comparator]
    for comparator in robustness_multi_parts
    if comparator in _learned_manifest_names
)
robustness_multi_required_predictions = []
for stress in robustness_multi_stress_list:
    for comparator in robustness_multi_parts:
        stem = ROBUSTNESS_MULTI_STEMS[comparator]
        robustness_multi_required_predictions.append(revision_root / 'predictions' / f'robustness_{stem}_{stress}_predictions.npz')

if '_restore_revision_artifacts_from_drive' in globals():
    _restore_revision_artifacts_from_drive(
        robustness_multi_clean_predictions + robustness_multi_contract_artifacts + robustness_multi_required_predictions + list(multi_robustness_outputs.values()),
        'Multi-comparator robustness artifacts',
    )

print('Multi-comparator clean prediction artifacts:')
for path in robustness_multi_clean_predictions:
    print(f'  {path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
print('Multi-comparator provenance contracts:')
for path in robustness_multi_contract_artifacts:
    print(f'  {path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
print('Multi-comparator required stress prediction artifacts:')
for path in robustness_multi_required_predictions:
    print(f'  {path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
robustness_multi_missing_clean = [path for path in robustness_multi_clean_predictions if not (path.exists() and path.stat().st_size > 0)]
robustness_multi_missing_contracts = [path for path in robustness_multi_contract_artifacts if not (path.exists() and path.stat().st_size > 0)]
robustness_multi_missing_predictions = [path for path in robustness_multi_required_predictions if not (path.exists() and path.stat().st_size > 0)]

if robustness_multi_missing_clean:
    raise FileNotFoundError(
        'Multi-comparator robustness cannot run without clean comparator predictions: '
        + '; '.join(path.as_posix() for path in robustness_multi_missing_clean)
    )
if robustness_multi_missing_contracts:
    raise FileNotFoundError(
        'Multi-comparator robustness provenance contracts are missing: '
        + '; '.join(path.as_posix() for path in robustness_multi_missing_contracts)
    )
for _comparator, _manifest_name in _learned_manifest_names.items():
    if _comparator not in robustness_multi_parts:
        continue
    _manifest_path = revision_root / 'manifests' / _manifest_name
    _manifest_payload = json.loads(_manifest_path.read_text(encoding='utf-8'))
    _checkpoint_contract = _manifest_payload.get('checkpoint_contract') or {}
    _checkpoint_rows = sorted(_checkpoint_contract.get('checkpoints') or [], key=lambda row: int(row.get('fold', -1)))
    if (
        _checkpoint_contract.get('status') != 'complete'
        or [int(row.get('fold', -1)) for row in _checkpoint_rows] != [1, 2, 3, 4, 5]
        or any(not row.get('sha256') for row in _checkpoint_rows)
    ):
        raise RuntimeError(
            f'{_comparator} baseline manifest lacks a complete five-fold checkpoint SHA contract. '
            'Rerun Notebook 04 in reuse mode to regenerate manifests without retraining.'
        )
if robustness_multi_missing_predictions:
    print('Missing stress predictions keep affected comparator/stress rows blocked unless generated first:')
    for path in robustness_multi_missing_predictions[:30]:
        print('  missing:', path)
    if len(robustness_multi_missing_predictions) > 30:
        print(f'  ... {len(robustness_multi_missing_predictions) - 30} more missing')

def _multi_robustness_outputs_current():
    required = list(multi_robustness_outputs.values())
    if not all(path.exists() and path.stat().st_size > 0 for path in required):
        return False, 'one or more requested output artifacts are missing'
    try:
        manifest = json.loads(multi_robustness_outputs['manifest'].read_text(encoding='utf-8'))
        pairwise = json.loads(multi_robustness_outputs['pairwise'].read_text(encoding='utf-8'))
        table = pd.read_csv(multi_robustness_outputs['table'])
        expected_rows = len(robustness_multi_stress_list) * (len(robustness_multi_parts) - 1) * len(robustness_multi_metric_list)
        expected_grid = {
            (stress, comparator, metric)
            for stress in robustness_multi_stress_list
            for comparator in robustness_multi_parts if comparator != 'full'
            for metric in robustness_multi_metric_list
        }
        observed_grid = set(zip(table['stress'], table['comparator'], table['metric']))
        if manifest.get('status') != 'complete' or pairwise.get('status') != 'complete':
            return False, 'manifest/pairwise status is not complete'
        expected_runner_sha = sha256_file(REPO_DIR / 'scripts/revision/21_robustness_multicomparator.py')
        if manifest.get('runner_sha256') != expected_runner_sha or pairwise.get('runner_sha256') != expected_runner_sha:
            return False, 'ledger runner SHA changed'
        expected_ci_scope = 'nominal_95_percentile_paired_record_bootstrap_unadjusted'
        FORENSIC_BOOTSTRAP_BINDING_CHECK = 'freeze_group_sidecar_v1'
        expected_bootstrap_unit = 'authenticated_source_patient_record'
        expected_training_scope = 'fixed_trained_folds_and_checkpoints_not_retrained_within_bootstrap'
        expected_class_support_policy = 'rank_calibration_omit_single_resampled_class_f1_keeps_all_labels_zero_division_zero'
        for label, payload in [('manifest', manifest), ('pairwise', pairwise)]:
            if payload.get('ci_scope') != expected_ci_scope:
                return False, f'{label} CI scope is stale or missing'
            if payload.get('bootstrap_unit') != expected_bootstrap_unit:
                return False, f'{label} bootstrap unit is stale or missing'
            if payload.get('training_variability_scope') != expected_training_scope:
                return False, f'{label} training-variability scope is stale or missing'
            if int(payload.get('metric_cache_schema_version', 0)) != 2:
                return False, f'{label} metric-cache schema is stale or missing'
            if payload.get('macro_class_support_policy') != expected_class_support_policy:
                return False, f'{label} macro class-support policy is stale or missing'
            independence = payload.get('bootstrap_independence_contract') or {}
            source_value = independence.get('source')
            source_path = Path(source_value) if source_value else None
            if source_path is not None and not source_path.is_absolute():
                source_path = REPO_DIR / source_path
            if (
                independence.get('unit') != expected_bootstrap_unit
                or independence.get('independence_contract') != 'physionet_ecg_arrhythmia_one_patient_per_record_v1'
                or independence.get('group_semantics_reference') != 'https://physionet.org/content/ecg-arrhythmia/1.0.0/'
                or not independence.get('group_sidecar_sha256')
                or independence.get('training_variability_scope') != expected_training_scope
                or source_path is None
                or not source_path.exists()
                or independence.get('source_sha256') != sha256_file(source_path)
            ):
                return False, f'{label} bootstrap independence contract is invalid'
        if set((manifest.get('stress_contracts') or {}).keys()) != set(robustness_multi_stress_list):
            return False, 'stress contracts are incomplete'
        artifact_rows = manifest.get('artifact_status') or []
        expected_artifact_rows = len(robustness_multi_parts) * (1 + len(robustness_multi_stress_list))
        if len(artifact_rows) != expected_artifact_rows or any(row.get('status') != 'ready' for row in artifact_rows):
            return False, 'prediction artifact provenance grid is incomplete or invalid'
        if manifest.get('output_profile') != ('canonical' if ROBUSTNESS_MULTI_OUTPUT_STEM == 'robustness_multicomparator' else manifest.get('output_profile')):
            return False, 'canonical output profile mismatch'
        if int(manifest.get('n_boot', 0)) != int(ROBUSTNESS_MULTI_N_BOOT):
            return False, 'bootstrap count changed'
        if manifest.get('stress_tests') != robustness_multi_stress_list or manifest.get('metrics') != robustness_multi_metric_list:
            return False, 'stress/metric grid changed'
        if len(table) != expected_rows or observed_grid != expected_grid:
            return False, f'expected complete grid with {expected_rows} rows'
        if (table['status'] != 'complete').any() or (table['n_boot_valid'].astype(int) < ROBUSTNESS_MULTI_N_BOOT).any():
            return False, 'one or more rows is incomplete/bootstrap-short'
        if (table['ci_scope'] != expected_ci_scope).any() or (table['bootstrap_unit'] != expected_bootstrap_unit).any():
            return False, 'one or more rows has a stale CI/bootstrap contract'
        if (table['training_variability_scope'] != expected_training_scope).any():
            return False, 'one or more rows has a stale training-variability scope'
        if (table['macro_class_support_policy'] != expected_class_support_policy).any():
            return False, 'one or more rows has a stale macro class-support policy'
        allowed_interpretations = {
            'full_nominal_95ci_more_favorable_change',
            'comparator_nominal_95ci_more_favorable_change',
            'nominal_95ci_inconclusive_change_difference',
        }
        if not set(table['interpretation']).issubset(allowed_interpretations):
            return False, 'one or more rows uses an unsafe legacy interpretation'
        current_oof = sha256_file(revision_root / 'predictions/oof_final_ema_predictions.npz')
        current_freeze = sha256_file(revision_root / 'manifests/oof_final_ema_freeze_manifest.json')
        contract = manifest.get('canonical_contract') or {}
        if contract.get('oof_sha256') != current_oof or contract.get('freeze_sha256') != current_freeze:
            return False, 'canonical OOF/freeze SHA changed'
        artifact_sha = manifest.get('artifact_sha256') or {}
        if artifact_sha.get('table') != sha256_file(multi_robustness_outputs['table']):
            return False, 'table SHA is not authenticated by the manifest'
        if artifact_sha.get('pairwise') != sha256_file(multi_robustness_outputs['pairwise']):
            return False, 'pairwise SHA is not authenticated by the manifest'
        return True, 'complete authenticated requested grid'
    except Exception as exc:
        return False, f'output validation failed: {exc!r}'

multi_outputs_ready, multi_outputs_reason = _multi_robustness_outputs_current()
print('Requested multi-comparator output contract:', multi_outputs_ready, '|', multi_outputs_reason)
multi_robustness_ran = False
print('Multi-comparator robustness command:', multi_robustness_command)
if not RUN_ROBUSTNESS_MULTICOMPARATOR:
    print('Multi-comparator robustness disabled. Existing artifacts, if present, remain usable.')
elif multi_outputs_ready and not FORCE_RERUN_ROBUSTNESS_MULTICOMPARATOR and not robustness_multi_missing_predictions:
    print('Requested multi-comparator robustness outputs are present and required predictions are complete; skipping aggregation rerun.')
else:
    if ROBUSTNESS_MULTI_RUN_PROFILE == 'canonical_resume':
        for stage_stress in robustness_multi_stress_list:
            safe_stage = ''.join(ch if ch.isalnum() or ch in {'_', '-'} else '_' for ch in stage_stress)
            stage_stem = f'robustness_multicomparator_{safe_stage}_canonical_stage'
            stage_summary = revision_root / 'metrics' / f'{stage_stem}_summary.csv'
            stage_pairwise = revision_root / 'metrics' / f'{stage_stem}_pairwise.json'
            stage_table = revision_root / 'tables' / f'table_{stage_stem}.csv'
            stage_manifest = revision_root / 'manifests' / f'{stage_stem}_manifest.json'
            stage_command = (
                'python -u scripts/revision/21_robustness_multicomparator.py '
                f'--comparators {ROBUSTNESS_MULTI_COMPARATORS} --stress-tests {stage_stress} '
                f'--metrics {ROBUSTNESS_MULTI_METRICS} --threshold 0.5 --n-bins 15 '
                f'--n-boot {ROBUSTNESS_MULTI_N_BOOT} --bootstrap-jobs {ROBUSTNESS_MULTI_BOOTSTRAP_JOBS} '
                f'--metric-cache-dir "{ROBUSTNESS_MULTI_METRIC_CACHE_DIR}" --reuse-metric-cache '
                f'--out-summary {stage_summary} --out-pairwise {stage_pairwise} '
                f'--out-table {stage_table} --out-manifest {stage_manifest}'
            )
            if ROBUSTNESS_MULTI_STRICT:
                stage_command += ' --strict'
            print(f'Canonical robustness stage: {stage_stress}')
            run(stage_command, log_path=f'reports/revision/logs/{stage_stem}.log')
            if 'MIRROR_REVISION_ROOT' in globals():
                run(
                    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
                    f'--refresh-existing-cache-dirs --refresh-existing-prefix metrics/robustness_multicomparator_metric_cache '
                    f'--mirror-root "{MIRROR_REVISION_ROOT}"',
                    log_path=f'reports/revision/logs/{stage_stem}_mirror_publish.log',
                )
        print('All six stress stages completed; assembling canonical cached ledger.')
    run(multi_robustness_command, log_path=f'reports/revision/logs/{ROBUSTNESS_MULTI_OUTPUT_STEM}.log')
    multi_robustness_ran = True
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size '
            f'--refresh-existing-cache-dirs --refresh-existing-prefix metrics/robustness_multicomparator_metric_cache '
            f'--mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path=f'reports/revision/logs/{ROBUSTNESS_MULTI_OUTPUT_STEM}_immediate_mirror_publish.log',
        )

multi_outputs_ready, multi_outputs_reason = _multi_robustness_outputs_current()
if ROBUSTNESS_MULTI_RUN_PROFILE in {'full_final', 'canonical_resume'} and not multi_outputs_ready:
    raise RuntimeError('Canonical six-stress robustness ledger is incomplete after aggregation: ' + multi_outputs_reason)
print('Post-run multi-comparator output contract:', multi_outputs_ready, '|', multi_outputs_reason)

for label, path in multi_robustness_outputs.items():
    print(f'{label:10s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

if multi_robustness_outputs['summary'].exists():
    multi_robustness_df = pd.read_csv(multi_robustness_outputs['summary'])
    _status_counts = multi_robustness_df.get('status', pd.Series(dtype=str)).value_counts(dropna=False).to_dict()
    status_counts = {str(key): int(value) for key, value in _status_counts.items()}
    print('Multi-comparator robustness row status counts:', status_counts)
    if 'interpretation' in multi_robustness_df.columns:
        print('Interpretation counts:', multi_robustness_df['interpretation'].value_counts(dropna=False).to_dict())
    display(multi_robustness_df.sort_values(['stress', 'comparator', 'metric']).reset_index(drop=True))
else:
    print('No multi-comparator robustness ledger present; broad robustness remains unsupported.')


## Claim Evidence Summary

This JSON is the rebuttal-facing interpretation boundary for HRV and robustness claims.


In [ ]:
# Self-contained summary preflight: this cell remains rerunnable after a Colab reconnect.
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from scripts.revision.common import git_commit, save_json, sha256_file

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
oof_prediction_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
hrv_schema_path = revision_root / 'hrv36_schema.csv'
hrv_summary_csv = revision_root / 'metrics' / 'hrv_domain_summary.csv'
hrv_prediction_path = revision_root / 'predictions' / 'hrv_only_oof_predictions.npz'
hrv_baseline_summary_path = revision_root / 'metrics' / 'hrv_only_baseline_summary.json'
hrv_domain_classifier_summary_path = revision_root / 'metrics' / 'hrv_domain_classifier_summary.json'
hrv_reuse_attestation_path = revision_root / 'metrics' / 'hrv_only_oof_reuse_attestation.json'
summary_required_paths = [
    oof_prediction_path, freeze_path, calibration_path, hrv_schema_path, hrv_summary_csv,
    hrv_prediction_path, hrv_baseline_summary_path, hrv_domain_classifier_summary_path,
    hrv_reuse_attestation_path,
]
if '_restore_verified_revision_artifact_05' in globals():
    for summary_artifact in summary_required_paths:
        if not summary_artifact.exists() or summary_artifact.stat().st_size == 0:
            relative = summary_artifact.relative_to(revision_root).as_posix()
            restore_status = _restore_verified_revision_artifact_05(relative, summary_artifact)
            print('Claim summary targeted restore:', relative, restore_status)
summary_missing_paths = [path.as_posix() for path in summary_required_paths if not path.exists() or path.stat().st_size == 0]
if summary_missing_paths:
    raise FileNotFoundError(
        'Claim Evidence Summary requires completed/restored Notebook 05 inputs: ' + '; '.join(summary_missing_paths)
    )
hrv_df = pd.read_csv(hrv_summary_csv)
hrv_statuses = dict(zip(hrv_df['analysis_name'], hrv_df['status']))
hrv_baseline_summary = json.loads(hrv_baseline_summary_path.read_text(encoding='utf-8'))
hrv_domain_classifier_summary = json.loads(hrv_domain_classifier_summary_path.read_text(encoding='utf-8'))
hrv_reuse_attestation = json.loads(hrv_reuse_attestation_path.read_text(encoding='utf-8'))
hrv_attestation_issues = []
if hrv_reuse_attestation.get('status') != 'verified':
    hrv_attestation_issues.append('status_not_verified')
if hrv_reuse_attestation.get('semantic_contract_match') is not True:
    hrv_attestation_issues.append('semantic_contract_not_matched')
if hrv_reuse_attestation.get('issues'):
    hrv_attestation_issues.extend(str(item) for item in hrv_reuse_attestation['issues'])
if hrv_reuse_attestation.get('current_oof_sha256') != sha256_file(oof_prediction_path):
    hrv_attestation_issues.append('attestation_current_oof_sha_mismatch')
if hrv_reuse_attestation.get('hrv_prediction_sha256') != sha256_file(hrv_prediction_path):
    hrv_attestation_issues.append('attestation_hrv_prediction_sha_mismatch')
if hrv_attestation_issues:
    raise RuntimeError(
        'HRV-only evidence lacks a valid current OOF semantic attestation. '
        'Rerun the HRV Domain Evidence cell before Claim Evidence Summary: '
        + '; '.join(dict.fromkeys(hrv_attestation_issues))
    )
hrv_only_complete = hrv_statuses.get('HRV-only baseline') == 'complete' and not hrv_attestation_issues
hrv_domain_complete = hrv_statuses.get('HRV domain classifier') == 'complete'
robustness_comparison_path = revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json'
robustness_summary_path = revision_root / 'metrics' / 'robustness_summary.csv'
from scripts.revision.robustness_profile_audit import select_best_profile

robustness_multi_audit = select_best_profile(
    revision_root,
    canonical_contract={
        'oof_sha256': sha256_file(oof_prediction_path),
        'freeze_sha256': sha256_file(freeze_path),
    },
    runner_path=Path('scripts/revision/21_robustness_multicomparator.py'),
    project_root=Path.cwd(),
)
robustness_multi_selected = robustness_multi_audit.get('selected') or {}
robustness_multi_selected_paths = robustness_multi_selected.get('paths') or {}
robustness_multi_summary_path = Path(robustness_multi_selected_paths['summary']) if robustness_multi_selected_paths.get('summary') else None
robustness_multi_pairwise_path = Path(robustness_multi_selected_paths['pairwise']) if robustness_multi_selected_paths.get('pairwise') else None
robustness_multi_manifest_path = Path(robustness_multi_selected_paths['manifest']) if robustness_multi_selected_paths.get('manifest') else None
robustness_payload = None
robustness_complete = False
if robustness_comparison_path.exists():
    robustness_payload = json.loads(robustness_comparison_path.read_text(encoding='utf-8'))
    robustness_complete = robustness_payload.get('status') is True
robustness_multi_complete = bool(robustness_multi_audit.get('complete'))
robustness_multi_interpretation_counts = robustness_multi_selected.get('interpretation_counts') or {}
if hrv_only_complete and hrv_domain_complete:
    hrv_status = 'hrv_only_and_domain_classifier_complete_duration_noise_blocked'
elif hrv_only_complete:
    hrv_status = 'hrv_only_complete_domain_classifier_blocked_or_missing_duration_noise_blocked'
else:
    hrv_status = 'blocked_or_incomplete'

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'hrv_schema': {'path': str(hrv_schema_path), 'sha256': sha256_file(hrv_schema_path)},
    'hrv_oof_reuse_attestation': {
        'path': str(hrv_reuse_attestation_path),
        'status': hrv_reuse_attestation.get('status'),
        'contract_status': 'exact_file_match' if hrv_reuse_attestation.get('exact_oof_sha_match') else 'semantic_input_match',
        'semantic_contract_match': hrv_reuse_attestation.get('semantic_contract_match'),
        'exact_oof_sha_match': hrv_reuse_attestation.get('exact_oof_sha_match'),
        'exact_freeze_sha_match': hrv_reuse_attestation.get('exact_freeze_sha_match'),
        'current_oof_semantic_sha256': hrv_reuse_attestation.get('current_oof_semantic_sha256'),
    },
    'hrv_status': hrv_status,
    'robustness_status': 'complete_perturbation_stress_tests' if robustness_complete else 'blocked_or_not_run',
    'robustness_multicomparator_status': robustness_multi_audit.get('status', 'blocked_or_not_run'),
    'robustness_multicomparator_audit': robustness_multi_audit,
    'robustness_artifacts': {
        'comparison_json': str(robustness_comparison_path) if robustness_comparison_path.exists() else None,
        'summary_csv': str(robustness_summary_path) if robustness_summary_path.exists() else None,
        'status': robustness_payload.get('status') if robustness_payload else False,
        'multicomparator_profile': robustness_multi_audit.get('selected_profile'),
        'multicomparator_evidence_tier': robustness_multi_selected.get('evidence_tier'),
        'multicomparator_metric_specific_ci_ready': robustness_multi_selected.get('metric_specific_ci_ready'),
        'multicomparator_summary_csv': str(robustness_multi_summary_path) if robustness_multi_summary_path else None,
        'multicomparator_pairwise_json': str(robustness_multi_pairwise_path) if robustness_multi_pairwise_path else None,
        'multicomparator_manifest': str(robustness_multi_manifest_path) if robustness_multi_manifest_path else None,
        'multicomparator_interpretation_counts': robustness_multi_interpretation_counts,
    },
    'hrv_metrics': {
        'hrv_only_roc_auc_macro': hrv_baseline_summary['metrics'].get('roc_auc_macro'),
        'hrv_only_pr_auc_macro': hrv_baseline_summary['metrics'].get('pr_auc_macro'),
        'hrv_only_f1_macro': hrv_baseline_summary['metrics'].get('f1_macro'),
        'domain_status': hrv_domain_classifier_summary.get('status', 'complete' if hrv_domain_complete else 'blocked_or_missing'),
        'domain_roc_auc_ovr_macro': (hrv_domain_classifier_summary.get('metrics') or {}).get('domain_roc_auc_ovr_macro'),
        'domain_balanced_accuracy': (hrv_domain_classifier_summary.get('metrics') or {}).get('balanced_accuracy'),
        'domain_interpretation': hrv_domain_classifier_summary.get('interpretation'),
        'domain_blocker': hrv_domain_classifier_summary.get('blocker'),
    },
    'claim_boundaries': [
        {
            'claim_id': 'C03',
            'claim_topic': 'HRV provides complementary rhythm descriptors',
            'current_status': 'partially_supported_by_hrv_only_only' if hrv_only_complete and not hrv_domain_complete else ('partially_supported_by_hrv_only_and_domain_classifier' if hrv_only_complete else 'not_supported_by_current_artifacts'),
            'safe_wording': 'HRV-only can be reported as a separate feature baseline. Domain-sensitivity evidence requires external HRV36 caches if the domain classifier is blocked. Duration/noise HRV sensitivity remains blocked.',
        },
        {
            'claim_id': 'A6/R2C3',
            'claim_topic': 'minimum robustness stress tests',
            'current_status': 'supported_by_perturbation_stress_artifacts' if robustness_complete else 'not_supported_by_current_artifacts',
            'safe_wording': (robustness_multi_audit.get('safe_wording') if robustness_multi_complete else 'Robustness tests must be run and reported as absolute/relative degradation before any robustness claim.'),
        },
    ],
}
summary_path = revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json'
save_json(summary_path, summary)
print('Wrote:', summary_path)
print(json.dumps(summary['hrv_metrics'], indent=2, sort_keys=True))


## Remaining Runner Guard

This cell records implemented runners and keeps any non-implemented follow-up work explicit.


In [ ]:
planned = {
    'HRV-only and domain analysis': {
        'status': 'implemented',
        'command': 'python scripts/revision/09_hrv_domain_analysis.py --oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz --freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json --expected-checkpoint-kind final_ema --chapman-hrv-cache /content/drive/MyDrive/ECG-Ramba/hrv36_N44186_C12_L5000_R{dataset_record_order_fingerprint}.npz',
    },
    'Full vs MiniRocket robustness stress tests': {
        'status': 'implemented',
        'command': 'python scripts/revision/12_robustness_stress.py --stress-tests snr20db,snr10db,snr5db,random_3_lead_dropout,precordial_dropout,resample_250hz',
    },
    'Duration/noise sensitivity for a complete HRV schema': {
        'status': 'deferred_full_hrv_retrain_required',
        'command': None,
    },
}

for name, item in planned.items():
    command = item['command']
    if command is None:
        print(f"{name:52s}: status={item['status']} command=none")
        continue
    script = Path(command.split()[1])
    print(f"{name:52s}: status={item['status']} implemented={script.exists()} command={command}")

if RUN_ROBUSTNESS_STRESS:
    stress_script = Path('scripts/revision/12_robustness_stress.py')
    if not stress_script.exists():
        raise RuntimeError('Robustness stress runner is missing: ' + str(stress_script))
else:
    print('Robustness stress execution disabled in this run. Existing artifacts, if restored, remain usable; otherwise robustness claims stay blocked.')


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
import inspect
if 'log_path' not in inspect.signature(run).parameters:
    raise RuntimeError('Notebook command helper run() was overwritten. Rerun the Setup cell before this inventory/mirror cell.')

run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_robustness_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --refresh-existing-cache-dirs --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_robustness_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'hrv_robustness_input_contract.json',
    revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_oof_reuse_attestation.json',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'metrics' / 'robustness_summary.csv',
    revision_root / 'tables' / 'table_robustness.csv',
    revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json',
    revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    revision_root / 'metrics' / 'robustness_full_vs_resnet_comparison.json',
    revision_root / 'metrics' / 'robustness_full_vs_raw_mamba_comparison.json',
    revision_root / 'metrics' / 'robustness_full_vs_transformer_comparison.json',
    revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
optional_outputs = [
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_fold_summary.csv',
    revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json',
    revision_root / 'metrics' / 'robustness_full_vs_minirocket_bootstrap_samples.csv',
    revision_root / 'manifests' / 'robustness_stress_manifest.json',
    revision_root / 'tables' / 'table_robustness_stress_metadata.csv',
    revision_root / 'tables' / 'table_robustness_artifacts.csv',
    revision_root / 'manifests' / 'comparator_stress_prediction_manifest.json',
    revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    revision_root / 'metrics' / 'robustness_full_vs_resnet_comparison.json',
    revision_root / 'metrics' / 'robustness_full_vs_raw_mamba_comparison.json',
    revision_root / 'metrics' / 'robustness_full_vs_transformer_comparison.json',
    revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
]

robustness_multicomparator_profile_outputs = []
for stem in [
    'robustness_multicomparator_reviewer_minimal',
    'robustness_multicomparator_full_screening',
    'robustness_multicomparator_core_final',
    'robustness_multicomparator_snr20db_final',
    'robustness_multicomparator_snr10db_final',
    'robustness_multicomparator_snr5db_final',
    'robustness_multicomparator_random_3_lead_dropout_final',
    'robustness_multicomparator_precordial_dropout_final',
    'robustness_multicomparator_resample_250hz_final',
]:
    robustness_multicomparator_profile_outputs.extend([
        revision_root / 'metrics' / f'{stem}_summary.csv',
        revision_root / 'metrics' / f'{stem}_pairwise.json',
        revision_root / 'tables' / f'table_{stem}.csv',
        revision_root / 'manifests' / f'{stem}_manifest.json',
    ])
    profile = stem.replace('robustness_multicomparator', '', 1).strip('_')
    for comparator in ['resnet', 'raw_mamba', 'transformer']:
        robustness_multicomparator_profile_outputs.append(
            revision_root / 'metrics' / f'robustness_full_vs_{comparator}_{profile}_comparison.json'
        )
for path in (revision_root / 'metrics').glob('robustness_multicomparator_*_final_summary.csv'):
    stem = path.name[:-len('_summary.csv')]
    robustness_multicomparator_profile_outputs.extend([
        revision_root / 'metrics' / f'{stem}_summary.csv',
        revision_root / 'metrics' / f'{stem}_pairwise.json',
        revision_root / 'tables' / f'table_{stem}.csv',
        revision_root / 'manifests' / f'{stem}_manifest.json',
    ])
    profile = stem.replace('robustness_multicomparator', '', 1).strip('_')
    for comparator in ['resnet', 'raw_mamba', 'transformer']:
        robustness_multicomparator_profile_outputs.append(
            revision_root / 'metrics' / f'robustness_full_vs_{comparator}_{profile}_comparison.json'
        )
# Keep order stable and avoid duplicate paths when a single-stress final file is discovered.
_seen_profile_outputs = set()
robustness_multicomparator_profile_outputs = [
    path for path in robustness_multicomparator_profile_outputs
    if not (path.as_posix() in _seen_profile_outputs or _seen_profile_outputs.add(path.as_posix()))
]
optional_outputs.extend(robustness_multicomparator_profile_outputs)
robustness_reference_outputs = [
    revision_root / 'predictions' / 'robustness_minirocket_clean_ref_predictions.npz',
    revision_root / 'manifests' / 'robustness_minirocket_heads_manifest.json',
]
stress_names = [
    'snr20db',
    'snr10db',
    'snr5db',
    'random_3_lead_dropout',
    'precordial_dropout',
    'resample_250hz',
]
robustness_prediction_outputs = []
for stress_name in stress_names:
    robustness_prediction_outputs.extend([
        revision_root / 'predictions' / f'robustness_full_{stress_name}_predictions.npz',
        revision_root / 'predictions' / f'robustness_minirocket_{stress_name}_predictions.npz',
        revision_root / 'predictions' / f'robustness_resnet1d_cnn_{stress_name}_predictions.npz',
        revision_root / 'predictions' / f'robustness_raw_mamba_{stress_name}_predictions.npz',
        revision_root / 'predictions' / f'robustness_transformer_ecg_{stress_name}_predictions.npz',
    ])
learned_comparator_checkpoints = []
for fold in range(1, 6):
    learned_comparator_checkpoints.extend([
        CANONICAL_CHECKPOINT_ROOT / 'resnet1d_cnn_checkpoints' / f'fold{fold}_resnet1d_cnn_final.pt',
        CANONICAL_CHECKPOINT_ROOT / 'raw_mamba_checkpoints' / f'fold{fold}_raw_mamba_final_ema.pt',
        CANONICAL_CHECKPOINT_ROOT / 'transformer_ecg_checkpoints' / f'fold{fold}_transformer_ecg_final.pt',
    ])


def _notebook05_restore_roots():
    candidates = []
    if 'MIRROR_REVISION_ROOT' in globals():
        candidates.append(Path(MIRROR_REVISION_ROOT))
    candidates.append(DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
    roots = []
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        key = candidate.as_posix()
        if key not in seen and candidate.exists():
            roots.append(candidate)
            seen.add(key)
    return roots


def _restore_notebook05_artifacts(paths, label):
    restored, reused, unresolved = [], [], []
    for destination in paths:
        destination = Path(destination)
        relative = destination.relative_to(revision_root).as_posix()
        status = _restore_verified_revision_artifact_05(relative, destination)
        if status == 'restored':
            restored.append(relative)
        elif status == 'reused':
            reused.append(relative)
        elif status == 'unverified_active':
            raise RuntimeError(f'Active artifact is absent from the canonical mirror manifest: {relative}')
        else:
            unresolved.append(relative)
    if restored or unresolved:
        print(f'{label} restore: restored={len(restored)} reused={len(reused)} unresolved={len(unresolved)}')
    return {'restored': restored, 'reused': reused, 'unresolved': unresolved}


_restore_notebook05_artifacts(expected_outputs + optional_outputs, 'Notebook 05 required/optional outputs')
_restore_notebook05_artifacts(robustness_reference_outputs + robustness_prediction_outputs, 'Notebook 05 robustness cache artifacts')

for path in expected_outputs:
    print(path, 'required exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in optional_outputs:
    print(path, 'optional exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in robustness_reference_outputs:
    print(path, 'robustness reference exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in robustness_prediction_outputs:
    print(path, 'robustness prediction exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in learned_comparator_checkpoints:
    print(path, 'checkpoint exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
missing_required = [str(path) for path in expected_outputs if not (path.exists() and path.stat().st_size > 0)]
missing_references = [str(path) for path in robustness_reference_outputs if not (path.exists() and path.stat().st_size > 0)]
missing_predictions = [str(path) for path in robustness_prediction_outputs if not (path.exists() and path.stat().st_size > 0)]
missing_checkpoints = [str(path) for path in learned_comparator_checkpoints if not (path.exists() and path.stat().st_size > 0)]
if missing_required:
    raise FileNotFoundError('Notebook 05 required outputs are missing: ' + '; '.join(missing_required))
if missing_references:
    print('WARNING: robustness reference cache artifacts are missing but aggregate evidence may still be reusable:', '; '.join(missing_references))
if missing_predictions:
    if globals().get('ROBUSTNESS_MULTI_RUN_PROFILE') in {'full_final', 'canonical_resume'}:
        raise FileNotFoundError('Canonical robustness requires all per-stress prediction caches: ' + '; '.join(missing_predictions))
    print('WARNING: robustness per-stress prediction cache artifacts are missing. Run comparator/full stress inference only for missing items if you need multi-comparator robustness:', '; '.join(missing_predictions))
if missing_checkpoints:
    print('WARNING: learned-comparator checkpoints are missing from the canonical mirror. Run only the missing Notebook 04 folds before comparator stress inference:', '; '.join(missing_checkpoints))

robustness_summary_path = revision_root / 'metrics' / 'robustness_summary.csv'
robustness_manifest_path = revision_root / 'manifests' / 'robustness_stress_manifest.json'
if robustness_summary_path.exists() and robustness_manifest_path.exists():
    robustness_summary = pd.read_csv(robustness_summary_path)
    expected_stresses = set(stress_names)
    expected_metrics = {'pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro'}
    actual_stresses = set(robustness_summary['stress_test'].astype(str))
    actual_metrics = set(robustness_summary['metric'].astype(str))
    if actual_stresses != expected_stresses:
        raise RuntimeError(f'Robustness summary stress set is incomplete: {sorted(actual_stresses)}')
    if actual_metrics != expected_metrics:
        raise RuntimeError(f'Robustness summary metric set is incomplete: {sorted(actual_metrics)}')
    if len(robustness_summary) != len(expected_stresses) * len(expected_metrics):
        raise RuntimeError(f'Robustness summary row count is incomplete: {len(robustness_summary)}')
    manifest_payload = json.loads(robustness_manifest_path.read_text(encoding='utf-8'))
    manifest_stresses = set(manifest_payload.get('args', {}).get('stress_tests', '').split(','))
    if manifest_stresses != expected_stresses:
        raise RuntimeError(f'Robustness manifest stress set is incomplete: {sorted(manifest_stresses)}')
    print('Notebook 05 robustness final summary validated: full six-stress table is present and manifest matches.')
    display(robustness_summary.sort_values(['stress_test', 'metric']).reset_index(drop=True))

multi_summary_path = revision_root / 'metrics' / 'robustness_multicomparator_summary.csv'
if multi_summary_path.exists():
    multi_df = pd.read_csv(multi_summary_path)
    print('Multi-comparator robustness status counts:', multi_df.get('status', pd.Series(dtype=str)).value_counts(dropna=False).to_dict())
    display(multi_df.sort_values(['stress', 'comparator', 'metric']).reset_index(drop=True))

for profile_summary_path in sorted((revision_root / 'metrics').glob('robustness_multicomparator_*_summary.csv')):
    if profile_summary_path.name == 'robustness_multicomparator_summary.csv':
        continue
    profile_df = pd.read_csv(profile_summary_path)
    print(f'Profile multi-comparator robustness summary: {profile_summary_path}')
    print('  rows:', len(profile_df), 'status_counts:', profile_df.get('status', pd.Series(dtype=str)).value_counts(dropna=False).to_dict())
    if 'interpretation' in profile_df.columns:
        print('  interpretation_counts:', profile_df['interpretation'].value_counts(dropna=False).to_dict())
    display(profile_df.sort_values(['stress', 'comparator', 'metric']).reset_index(drop=True))

if (revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json').exists():
    if missing_references or missing_predictions:
        print('Notebook 05 complete with aggregate robustness evidence restored/validated. Some reusable prediction cache artifacts are missing locally; rerun only missing stress inference when needed.')
    else:
        print('Notebook 05 complete. HRV/domain evidence and perturbation robustness artifacts are present.')
    print('Use only named stress/metric/comparator robustness statements supported by pointwise paired degradation CIs; do not infer family-wise significance across the full grid.')
    print('Learned-comparator robustness is usable only for profiles whose robustness_multicomparator_* outputs are present and regenerated into Notebook 07.')
else:
    print('Notebook 05 complete. HRV-only evidence is implemented; HRV domain evidence is complete only when external HRV36 caches exist. Robustness claims remain blocked until perturbation inference artifacts exist.')
